In [ ]:
# 1. Cài đặt condacolab để tạo ra một môi trường conda ổn định
!pip install -q condacolab
import condacolab
condacolab.install()

# 2. Cài đặt các thư viện hệ thống và Python cần thiết vào môi trường conda
# Lưu ý: Các thư viện sẽ được cài bằng pip bên trong môi trường conda đã được thiết lập
!apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig

!pip install swig
!pip install wrds
!pip install pyportfolioopt
!pip install quantstats
!pip install pandas_market_calendars

# Cài đặt phiên bản mới nhất của FinRL từ GitHub
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

# Cài đặt PyTorch tương thích với GPU
# Luôn đặt lệnh này ở cuối để đảm bảo nó ghi đè lên các phiên bản khác nếu có xung đột
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!pip install sb3-contrib


In [ ]:
# Cài đặt lại Numpy bản ổn định cũ
!pip install "numpy<2.0.0" --force-reinstall

# TRAINING


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import time
import random
import torch as th
import torch.nn as nn
import os
import gymnasium as gym
from collections import deque, defaultdict
from tqdm.auto import tqdm
from copy import deepcopy

# --- FinRL & SB3 Imports ---
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.main import check_and_make_directories
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
)
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.utils import set_random_seed, explained_variance
from stable_baselines3.common.logger import configure
from stable_baselines3.common.callbacks import BaseCallback
from sb3_contrib import RecurrentPPO

# Tạo thư mục cần thiết
check_and_make_directories([DATA_SAVE_DIR, TRAINED_MODEL_DIR, TENSORBOARD_LOG_DIR, RESULTS_DIR])

# =============================================================================
# 1. CONFIGURATION
# =============================================================================
print("="*80)
print("DAPO ASYMMETRIC CLIP (EXTENDED TEST PERIOD)")
print("="*80)

# --- DAPO Config ---
# Asymmetric Clipping: Cho phép policy cập nhật mạnh hơn về hướng tốt
DAPO_EPSILON_LOW = 0.2
DAPO_EPSILON_HIGH = 0.28

# --- Data Config ---
TRAIN_PATH = '/kaggle/input/file-du-lieu-vi-embedding/train2025_sentiment_with_emb.csv'
TEST_PATH = '/kaggle/input/test-moi/test2025_sentiment_with_emb2.csv'
EMBEDDING_DIM = 1024

# --- Experiment Config ---
FIXED_WINDOW_SIZE = 15 
EMBEDDING_MODES = ["Decay30"] 
TRAIN_SEEDS = [30,31,32]
BACKTEST_SEEDS = [100]
NUM_CPU = 4  

# --- Reward Config ---
TAIL_CONFIG = {'enable': False, 'tail_lambda': 0.5, 'window': 30, 'alpha': 0.01}
CVAR_CONFIG = {'enable': True, 'weight': 0.5, 'window': 90, 'alpha': 0.05}

# --- Model Config (Standard PPO Params) ---
TOTAL_TIMESTEPS = 1_000_000 
PPO_PARAMS = {
    "n_steps": 2048, 
    "ent_coef": 0.01,
    "learning_rate": 0.00025, # Default LR
    "batch_size": 128,
    "target_kl": None         # No KL Penalty
}

# --- Early Stopping Config ---
EARLY_STOPPING_ENABLED = True
EARLY_STOPPING_PATIENCE = 45 
EARLY_STOPPING_MIN_DELTA = 0.005
EARLY_STOPPING_WINDOW = 15

# =============================================================================
# 2. CUSTOM CLASS: DAPO RECURRENT PPO
# =============================================================================

class DAPORecurrentPPO(RecurrentPPO):
    """
    Kế thừa RecurrentPPO nhưng thay đổi hàm train để hỗ trợ Asymmetric Clipping của DAPO.
    """
    def __init__(self, *args, epsilon_low=0.2, epsilon_high=0.2, **kwargs):
        super().__init__(*args, **kwargs)
        self.epsilon_low = epsilon_low
        self.epsilon_high = epsilon_high

    def train(self) -> None:
        """
        Ghi đè hàm train để thực hiện Asymmetric Clipping.
        """
        self.policy.set_training_mode(True)
        self._update_learning_rate(self.policy.optimizer)

        # Compute current clip range
        clip_range = self.clip_range(self._current_progress_remaining)
        if self.clip_range_vf is not None:
            clip_range_vf = self.clip_range_vf(self._current_progress_remaining)

        entropy_losses = []
        pg_losses, value_losses = [], []
        clip_fractions = []

        continue_training = True
        
        # Train for n_epochs epochs
        for epoch in range(self.n_epochs):
            approx_kl_divs = []
            # Do sampling
            for rollout_data in self.rollout_buffer.get(self.batch_size):
                actions = rollout_data.actions
                if isinstance(self.action_space, gym.spaces.Discrete):
                    actions = rollout_data.actions.long().flatten()

                # Forward pass
                values, log_prob, entropy = self.policy.evaluate_actions(
                    rollout_data.observations,
                    actions,
                    rollout_data.lstm_states,
                    rollout_data.episode_starts,
                )

                values = values.flatten()
                
                # Normalize advantage
                advantages = rollout_data.advantages
                if self.normalize_advantage:
                    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

                # Ratio calculation
                ratio = th.exp(log_prob - rollout_data.old_log_prob)

                # --- DAPO MODIFICATION: ASYMMETRIC CLIPPING ---
                clip_min = 1.0 - self.epsilon_low
                clip_max = 1.0 + self.epsilon_high
                
                policy_loss_1 = advantages * ratio
                policy_loss_2 = advantages * th.clamp(ratio, clip_min, clip_max)
                policy_loss = -th.min(policy_loss_1, policy_loss_2).mean()
                # -----------------------------------------------

                pg_losses.append(policy_loss.item())
                clip_fraction = th.mean((th.abs(ratio - 1) > clip_range).float()).item()
                clip_fractions.append(clip_fraction)

                if self.clip_range_vf is None:
                    values_pred = values
                else:
                    values_pred = rollout_data.old_values + th.clamp(
                        values - rollout_data.old_values, -clip_range_vf, clip_range_vf
                    )

                value_loss = th.nn.functional.mse_loss(rollout_data.returns, values_pred)
                value_losses.append(value_loss.item())

                if entropy is None:
                    entropy_loss = -th.mean(-log_prob)
                else:
                    entropy_loss = -th.mean(entropy)

                entropy_losses.append(entropy_loss.item())

                loss = policy_loss + self.ent_coef * entropy_loss + self.vf_coef * value_loss

                with th.no_grad():
                    log_ratio = log_prob - rollout_data.old_log_prob
                    approx_kl_div = th.mean((th.exp(log_ratio) - 1) - log_ratio).cpu().numpy()
                    approx_kl_divs.append(approx_kl_div)

                if self.target_kl is not None and approx_kl_div > 1.5 * self.target_kl:
                    continue_training = False
                    if self.verbose >= 1:
                        print(f"Early stopping at step {epoch} due to reaching max kl: {approx_kl_div:.2f}")
                    break

                self.policy.optimizer.zero_grad()
                loss.backward()
                th.nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
                self.policy.optimizer.step()

            self._n_updates += 1
            if not continue_training:
                break

        explained_var = explained_variance(self.rollout_buffer.values.flatten(), self.rollout_buffer.returns.flatten())

        self.logger.record("train/entropy_loss", np.mean(entropy_losses))
        self.logger.record("train/policy_gradient_loss", np.mean(pg_losses))
        self.logger.record("train/value_loss", np.mean(value_losses))
        self.logger.record("train/approx_kl", np.mean(approx_kl_divs))
        self.logger.record("train/clip_fraction", np.mean(clip_fractions))
        self.logger.record("train/loss", loss.item())
        self.logger.record("train/explained_variance", explained_var)
        if hasattr(self, "lr_schedule"):
            self.logger.record("train/learning_rate", self.lr_schedule(self._current_progress_remaining))
        self.logger.record("train/n_updates", self._n_updates)
        self.logger.record("train/clip_range", clip_range)
        if self.clip_range_vf is not None:
            self.logger.record("train/clip_range_vf", clip_range_vf)

# =============================================================================
# 3. DATA PROCESSING
# =============================================================================
def clean_embedding_str(x):
    try:
        if isinstance(x, str):
            x_clean = x.replace('[', '').replace(']', '').replace('\n', ' ').strip()
            arr = np.fromstring(x_clean, sep=' ', dtype=np.float32)
            if arr.shape[0] == EMBEDDING_DIM: return arr
            elif arr.shape[0] > EMBEDDING_DIM: return arr[:EMBEDDING_DIM]
            else: return np.pad(arr, (0, EMBEDDING_DIM - arr.shape[0]))
        elif isinstance(x, (list, np.ndarray)):
            return np.array(x, dtype=np.float32)
        return np.zeros(EMBEDDING_DIM, dtype=np.float32)
    except:
        return np.zeros(EMBEDDING_DIM, dtype=np.float32)

def process_dataframe_csv(df_path):
    print(f"Processing: {df_path}")
    df = pd.read_csv(df_path)
    
    col_map = {}
    for col in df.columns:
        c_lower = col.lower()
        if 'date' in c_lower: col_map[col] = 'date'
        elif 'ticker' in c_lower or 'tic' in c_lower: col_map[col] = 'tic'
        elif 'embedding' in c_lower: col_map[col] = 'news_embedding'
        elif col in INDICATORS + ['open', 'high', 'low', 'close', 'volume', 'turbulence']: 
            col_map[col] = col
    df.rename(columns=col_map, inplace=True)
    
    df['date'] = pd.to_datetime(df['date'], dayfirst=False, errors='coerce')
    df = df.dropna(subset=['date']).sort_values(['date', 'tic']).reset_index(drop=True)
    
    if 'news_embedding' in df.columns:
        df['news_embedding'] = df['news_embedding'].apply(clean_embedding_str)
    else:
        df['news_embedding'] = [np.zeros(EMBEDDING_DIM, dtype=np.float32)] * len(df)

    # Fill missing
    all_dates = df['date'].unique()
    all_tickers = df['tic'].unique()
    multi_index = pd.MultiIndex.from_product([all_dates, all_tickers], names=['date', 'tic'])
    df = pd.merge(pd.DataFrame(index=multi_index).reset_index(), df, on=['date', 'tic'], how='left')
    df = df.sort_values(['tic', 'date'])
    
    num_cols = ['open', 'high', 'low', 'close', 'volume'] + INDICATORS + ['turbulence']
    target = [c for c in num_cols if c in df.columns]
    df[target] = df.groupby('tic')[target].transform(lambda x: x.ffill().bfill().fillna(0))
    
    # Fill embedding nulls
    null_mask = df['news_embedding'].isnull()
    if null_mask.any():
         df.loc[null_mask, 'news_embedding'] = df.loc[null_mask, 'news_embedding'].apply(lambda x: np.zeros(EMBEDDING_DIM, dtype=np.float32))
            
    return df

# --- Load Data & Split ---
try:
    # 1. Load full datasets first
    raw_train = process_dataframe_csv(TRAIN_PATH)
    raw_test = process_dataframe_csv(TEST_PATH)
    
    # 2. Combine to handle date shifting
    full_data = pd.concat([raw_train, raw_test]).sort_values(['date', 'tic']).reset_index(drop=True)
    
    # 3. Calculate new split point (Shift 1 year from Train to Test)
    # Tìm ngày bắt đầu của Test set hiện tại
    original_test_start = raw_test['date'].min()
    print(f"Original Test Start: {original_test_start}")
    
    # Dời về trước 365 ngày
    new_split_date = original_test_start - pd.Timedelta(days=0)
    print(f"New Split Date (Shifted 1 year back): {new_split_date}")
    
    # 4. Re-split
    train = full_data[full_data['date'] < new_split_date].copy()
    trade = full_data[full_data['date'] >= new_split_date].copy()
    
    print(f"New Train Range: {train['date'].min()} -> {train['date'].max()} ({len(train)} rows)")
    print(f"New Test Range:  {trade['date'].min()} -> {trade['date'].max()} ({len(trade)} rows)")

except Exception as e:
    print(f"❌ Data Error: {e}")
    # raise e

# --- Create Global Variables ---
train_dates = sorted(train['date'].unique())
trade_dates = sorted(trade['date'].unique())
train['day_index'] = train['date'].map({d: i for i, d in enumerate(train_dates)})
trade['day_index'] = trade['date'].map({d: i for i, d in enumerate(trade_dates)})

train_env_df = train.set_index('day_index')
trade_env_df = trade.set_index('day_index')
stock_dimension = train['tic'].nunique()
print(f"✓ Stock Dimension: {stock_dimension}")

# --- Create Embeddings ---
def get_daily_mean(df):
    return df.groupby('day_index')['news_embedding'].apply(lambda x: np.mean(np.vstack(x.values), axis=0))

train_daily = get_daily_mean(train)
trade_daily = get_daily_mean(trade)

TRAIN_EMBEDDINGS = {}
TRADE_EMBEDDINGS = {}
TRAIN_EMBEDDINGS['Decay30'] = {i: r.values.astype(np.float32) for i, r in pd.DataFrame(train_daily.tolist()).ewm(span=30, adjust=False).mean().iterrows()}
TRADE_EMBEDDINGS['Decay30'] = {i: r.values.astype(np.float32) for i, r in pd.DataFrame(trade_daily.tolist()).ewm(span=30, adjust=False).mean().iterrows()}


# =============================================================================
# 4. ENVIRONMENT CLASSES
# =============================================================================

class StockTradingEnvEmbeddingAndReward(StockTradingEnv):
    def __init__(self, df, embedding_dict, **kwargs):
        self.embedding_dict = embedding_dict
        self.embedding_dim = EMBEDDING_DIM
        
        # Tail Risk Config
        self.enable_tail_risk_penalty = TAIL_CONFIG['enable']
        self.tail_lambda = TAIL_CONFIG['tail_lambda']
        self.tail_window = TAIL_CONFIG['window']
        self.tail_alpha = TAIL_CONFIG['alpha']
        
        # CVaR Diversification Config
        self.enable_cvar_div_penalty = CVAR_CONFIG['enable']
        self.cvar_div_penalty_weight = CVAR_CONFIG['weight']
        self.cvar_window = CVAR_CONFIG['window']
        self.cvar_alpha = CVAR_CONFIG['alpha']
        
        self.full_df = df.copy()
        temp_df = self.full_df.copy()
        if 'day_index' not in temp_df.columns and temp_df.index.name == 'day_index':
            temp_df = temp_df.reset_index()
        self.pivot_returns = temp_df.pivot(index='day_index', columns='tic', values='close').pct_change().fillna(0)
        
        super().__init__(df=df, **kwargs)
        self.action_space = gym.spaces.Box(low=-1, high=1, shape=(self.action_space.shape[0],), dtype=np.float32)
        self.asset_history = [self.initial_amount]

    def _get_daily_embedding(self):
        return self.embedding_dict.get(self.day, np.zeros(self.embedding_dim, dtype=np.float32))

    def _calculate_tail_risk_penalty(self, begin_asset):
        if not self.enable_tail_risk_penalty: 
            return 0.0
        try:
            if len(self.asset_history) < 2: 
                return 0.0
            arr = np.array(self.asset_history)
            ret = np.diff(arr)/arr[:-1]
            if len(ret) == 0: 
                return 0.0
            win = min(len(ret), self.tail_window)
            recent = ret[-win:]
            var = np.percentile(recent, self.tail_alpha*100)
            tail = recent[recent <= var]
            cvar = tail.mean() if len(tail) > 0 else var
            return -cvar * begin_asset if cvar < 0 else 0.0
        except: 
            return 0.0

    def _calculate_cvar_diversification_penalty(self):
        if not self.enable_cvar_div_penalty:
            return 0.0
        try:
            if self.day < self.cvar_window: 
                return 0.0
            hist_ret = self.pivot_returns.loc[self.day-self.cvar_window : self.day-1].values
            if hist_ret.shape[0] < self.cvar_window: 
                return 0.0
            
            shares = np.array(self.state[1+self.stock_dim : 1+2*self.stock_dim])
            prices = np.array(self.state[1 : 1+self.stock_dim])
            pos_val = shares * prices
            total = np.sum(pos_val) + self.state[0]
            
            if total <= 0 or np.sum(pos_val/total) < 1e-5: 
                return 0.0
            
            w = pos_val / total
            var_i = np.percentile(hist_ret, self.cvar_alpha*100, axis=0)
            cvar_i = []
            for i in range(self.stock_dim):
                tl = hist_ret[:,i][hist_ret[:,i] <= var_i[i]]
                val = tl.mean() if len(tl)>0 else var_i[i]
                cvar_i.append(-val)
            
            weighted_risk = np.sum(w * np.maximum(np.array(cvar_i), 1e-6))
            port_ret = np.dot(hist_ret, w)
            var_p = np.percentile(port_ret, self.cvar_alpha*100)
            tl_p = port_ret[port_ret <= var_p]
            cvar_p = max(-(tl_p.mean() if len(tl_p)>0 else var_p), 1e-6)
            dr = weighted_risk / cvar_p
            
            return self.cvar_div_penalty_weight * (1.0 / (dr + 1e-8))
        except: 
            return 0.0

    def reset(self, *, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        emb = self._get_daily_embedding()
        full_obs = np.concatenate((np.array(obs, dtype=np.float32).flatten(), emb))
        self.state = full_obs
        
        fin_dim = 1 + 2 * self.stock_dim + len(INDICATORS) * self.stock_dim
        current_fin_state = self.state[:fin_dim]
        ia = current_fin_state[0] + sum(np.array(current_fin_state[1:(self.stock_dim+1)]) * np.array(current_fin_state[(self.stock_dim+1):(self.stock_dim*2+1)]))
        self.asset_history = [ia]
        return full_obs, info

    def step(self, actions):
        fin_dim = 1 + 2 * self.stock_dim + len(INDICATORS) * self.stock_dim
        current_fin_state = self.state[:fin_dim]
        begin_asset = current_fin_state[0] + sum(
            np.array(current_fin_state[1:(self.stock_dim+1)]) * 
            np.array(current_fin_state[(self.stock_dim+1):(self.stock_dim*2+1)])
        )

        orig_scale = self.reward_scaling
        self.reward_scaling = 1.0
        next_state_fin, reward, done, truncated, info = super().step(actions)
        self.reward_scaling = orig_scale
        
        emb = self._get_daily_embedding()
        full_next_state = np.concatenate((np.array(next_state_fin, dtype=np.float32).flatten(), emb))
        self.state = full_next_state
        
        current_fin_state_new = self.state[:fin_dim]
        end_asset = current_fin_state_new[0] + sum(
            np.array(current_fin_state_new[1:(self.stock_dim+1)]) * 
            np.array(current_fin_state_new[(self.stock_dim+1):(self.stock_dim*2+1)])
        )
        self.asset_history.append(end_asset)
        
        base_profit = end_asset - begin_asset
        tail_p = self._calculate_tail_risk_penalty(begin_asset)
        cvar_div_p = self._calculate_cvar_diversification_penalty()
        
        final_reward = (base_profit - self.tail_lambda * tail_p) * self.reward_scaling
        final_reward = final_reward - cvar_div_p
        
        return full_next_state, final_reward, done, truncated, info

class EnhancedStockTradingEnvLSTM(StockTradingEnvEmbeddingAndReward):
    def __init__(self, df, window_size, embedding_dict, **kwargs):
        self.window_size = window_size
        
        if 'single_day_state_space' in kwargs:
            self.single_day_dim = kwargs.pop('single_day_state_space')
        else:
            n_stocks = kwargs.get('stock_dim', 30)
            n_inds = len(kwargs.get('tech_indicator_list', []))
            base_dim = 1 + 2 * n_stocks + n_inds * n_stocks
            self.single_day_dim = base_dim + EMBEDDING_DIM
        
        total_state_space = self.window_size * self.single_day_dim
        kwargs['state_space'] = total_state_space
        
        super().__init__(df=df, embedding_dict=embedding_dict, **kwargs)
        
        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, 
            shape=(total_state_space,), 
            dtype=np.float32
        )
        self.state_memory_window = deque(maxlen=self.window_size)

    def reset(self, *, seed=None, options=None):
        self.state_memory_window.clear()
        s, _ = super().reset(seed=seed, options=options)
        for _ in range(self.window_size): 
            self.state_memory_window.append(np.array(s, dtype=np.float32))
        return np.concatenate(list(self.state_memory_window)), {}

    def step(self, actions):
        next_s, r, d, t, i = super().step(actions)
        self.state_memory_window.append(np.array(next_s, dtype=np.float32))
        return np.concatenate(list(self.state_memory_window)), r, d, t, i

# =============================================================================
# 5. CALLBACKS
# =============================================================================

class SingleModelEarlyStoppingCallback(BaseCallback):
    def __init__(self, total_timesteps, patience=35, min_delta=0.005, window_len=15, 
                 save_path_prefix="", verbose=1):
        super().__init__(verbose)
        self.total_timesteps = total_timesteps
        self.patience = patience
        self.min_delta = min_delta
        self.window_len = window_len
        self.save_path_prefix = save_path_prefix
        
        self.pbar = None
        self.episode_rewards = []
        self.best_mean_reward = -np.inf
        self.episodes_without_improvement = 0
        self.early_stop_triggered = False

    def _on_training_start(self):
        self.pbar = tqdm(total=self.total_timesteps, desc="Training Progress") 

    def _on_step(self):
        self.pbar.update(self.training_env.num_envs)
        
        for i in range(self.training_env.num_envs):
            if self.locals['dones'][i]:
                if 'episode' in self.locals['infos'][i]:
                    ep_reward = self.locals['infos'][i]['episode']['r']
                    self.episode_rewards.append(ep_reward)
                    
                    if len(self.episode_rewards) >= self.window_len:
                        recent_mean = np.mean(self.episode_rewards[-self.window_len:])
                        self.pbar.set_postfix({
                            'mean_15': f'{recent_mean:.2f}',
                            'best': f'{self.best_mean_reward:.2f}',
                            'no_imp': self.episodes_without_improvement,
                        })
                        
                        if recent_mean > self.best_mean_reward + self.min_delta:
                            self.best_mean_reward = recent_mean
                            self.episodes_without_improvement = 0
                        else:
                            self.episodes_without_improvement += 1
                        
                        if self.episodes_without_improvement >= self.patience:
                            self.early_stop_triggered = True
                            self.model.save(f"{self.save_path_prefix}_FINAL")
                            tqdm.write(f"\n⚠ Patience exhausted! Saved FINAL model and stopping training.")
                            return False 

        return True

    def _on_training_end(self):
        if self.pbar: self.pbar.close()
        if not self.early_stop_triggered:
             self.model.save(f"{self.save_path_prefix}_FINAL")
             print("✓ Completed max timesteps. Saved FINAL model.")

# =============================================================================
# 6. MAIN EXECUTION LOOP
# =============================================================================

buy_cost_list = sell_cost_list = [0.001] * stock_dimension
num_stock_shares = [0] * stock_dimension
base_dim = 1 + 2 * stock_dimension + len(INDICATORS) * stock_dimension
single_day_dim_with_emb = base_dim + EMBEDDING_DIM

base_env_kwargs = {
    "hmax": 200, "initial_amount": 1000000, "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list, "sell_cost_pct": sell_cost_list,
    "stock_dim": stock_dimension, "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension, "reward_scaling": 1e-4,
    "single_day_state_space": single_day_dim_with_emb 
}

def make_env(rank, seed=0, emb_mode="Decay30"):
    def _init():
        env = EnhancedStockTradingEnvLSTM(
            df=train_env_df, 
            window_size=FIXED_WINDOW_SIZE, 
            embedding_dict=TRAIN_EMBEDDINGS[emb_mode],
            **base_env_kwargs
        )
        env.reset(seed=seed + rank) 
        return env
    return _init

def train_lstm_model_comparison(emb_mode, seed):
    conf_name = f"DAPO_PPO_LSTM_{emb_mode}_Seed{seed}"
    save_prefix = f"{TRAINED_MODEL_DIR}/{conf_name}"
    print(f"\n>>> TRAINING: {conf_name}")
    
    th.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    
    env_train = make_vec_env(
        make_env(0, seed, emb_mode),
        n_envs=NUM_CPU,              
        seed=seed,
        vec_env_cls=SubprocVecEnv    
    )
    
    # Default Policy Config
    current_policy_kwargs = {
        "ortho_init": True, 
        "lstm_hidden_size": 256,
        "n_lstm_layers": 1,
        "activation_fn": nn.Tanh
    }

    logger = configure(f"{TENSORBOARD_LOG_DIR}/{conf_name}", ["stdout", "tensorboard"])
    
    # USE DAPORecurrentPPO instead of RecurrentPPO
    model = DAPORecurrentPPO(
        "MlpLstmPolicy", 
        env_train, 
        verbose=0, 
        policy_kwargs=current_policy_kwargs,
        epsilon_low=DAPO_EPSILON_LOW,   # Param DAPO
        epsilon_high=DAPO_EPSILON_HIGH, # Param DAPO
        **PPO_PARAMS
    )
    model.set_logger(logger)
    
    my_callback = SingleModelEarlyStoppingCallback(
        total_timesteps=TOTAL_TIMESTEPS,
        patience=EARLY_STOPPING_PATIENCE,
        min_delta=EARLY_STOPPING_MIN_DELTA,
        window_len=EARLY_STOPPING_WINDOW,
        save_path_prefix=save_prefix
    )
    
    try:
        model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=my_callback)
        env_train.close()
        return save_prefix 
    except Exception as e:
        print(f"    ❌ Error: {e}")
        import traceback
        traceback.print_exc()
        env_train.close()
        return None

def backtest_lstm_manual(model, emb_mode, seed):
    th.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    
    env = EnhancedStockTradingEnvLSTM(
        df=trade_env_df, 
        window_size=FIXED_WINDOW_SIZE, 
        embedding_dict=TRADE_EMBEDDINGS[emb_mode], 
        **base_env_kwargs
    )
    
    obs, _ = env.reset()
    lstm_states = None
    done = False
    
    while not done:
        input_states = lstm_states 
        # Deterministic=True for backtesting
        action, lstm_states = model.predict(obs, state=input_states, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        if done or truncated: break
        
    account_values = env.asset_history
    test_dates = trade['date'].unique()
    
    if len(account_values) > len(test_dates):
        account_values = account_values[:len(test_dates)]
    
    df_res = pd.DataFrame({'account_value': account_values})
    df_res['date'] = test_dates[:len(account_values)]
    return df_res

# Run Comparison
results = {}
for mode in EMBEDDING_MODES:
    for seed in TRAIN_SEEDS:
        prefix = train_lstm_model_comparison(mode, seed)
        
        if prefix:
            ckpt = "FINAL"
            model_path = f"{prefix}_{ckpt}.zip"
            
            if os.path.exists(model_path):
                print(f"   ► Backtesting: {model_path}")
                try:
                    # Load using DAPO class
                    loaded_model = DAPORecurrentPPO.load(model_path)
                    
                    for bt_seed in BACKTEST_SEEDS:
                        df_result = backtest_lstm_manual(loaded_model, mode, bt_seed)
                        if not df_result.empty:
                            final_val = df_result['account_value'].iloc[-1]
                            ret = (final_val - 1000000) / 1000000 * 100
                            
                            key = f"{mode}_Seed{seed}_{ckpt}"
                            results[key] = df_result
                            print(f"      -> Result: ${final_val:,.2f} ({ret:.2f}%)")
                except Exception as e:
                    print(f"      ❌ Backtest Failed: {e}")
            else:
                print(f"   ⚠ Model file not found: {model_path}")

print("\n✅ ALL DONE.")

# BACKTEST OPEN T+1

In [1]:
# =============================================================================
# BACKTEST FIXED: 2-LAYER ENSEMBLE (LOGGING + DETERMINISTIC CPU)
# =============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import shutil
import zipfile
import gymnasium as gym
from sb3_contrib import RecurrentPPO
from collections import deque
from itertools import product
import warnings
import random
import torch

warnings.filterwarnings("ignore")

# --- FinRL Imports ---
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.config import INDICATORS

# =============================================================================
# 1. CẤU HÌNH DETERMINISTIC & PARAMETERS
# =============================================================================
SEED = 100

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"✅ Random Seed Set: {seed}")

set_seed(SEED)

# --- CẤU HÌNH DYNAMIC ENSEMBLE ---
ENABLE_DYNAMIC_ENSEMBLE = True

# --- GRID SEARCH PARAMETERS ---
# Lớp 2: Chọn model ngắn hạn (Trading)
GRID_L2_LOOKBACK_MONTHS = [1]    
GRID_L2_TOP_K_MODELS    = [1]    

# Lớp 1: Lọc Pool dài hạn (Filter)
GRID_L1_FILTER_MONTHS   = [10]    
GRID_L1_TOP_N_POOL      = [10]    

# --- Cấu hình dữ liệu ---
TRAIN_PATH = '/kaggle/input/file-du-lieu-vi-embedding/train2025_sentiment_with_emb.csv'
TEST_PATH = '/kaggle/input/test-moi/test2025_sentiment_with_emb2.csv'
EMBEDDING_DIM = 1024
FIXED_WINDOW_SIZE = 15
EMBEDDING_MODES = ["Decay30", "Rolling30", "Basic"]

# --- MODEL DIRECTORY ---
TRAINED_MODEL_DIRS = [
    "/kaggle/input/dulieu-hmax200-2022",
    "/kaggle/input/modeltest",
    "./TRAINED_MODEL_DIR"
]

TRAINED_MODEL_DIR = None
for path in TRAINED_MODEL_DIRS:
    if os.path.exists(path):
        TRAINED_MODEL_DIR = path
        print(f"✓ Found model directory: {TRAINED_MODEL_DIR}")
        break

if TRAINED_MODEL_DIR is None:
    print("⚠ CẢNH BÁO: Không tìm thấy thư mục model! Sử dụng mặc định.")
    TRAINED_MODEL_DIR = "/kaggle/working/trained_models"

# =============================================================================
# 2. XỬ LÝ DỮ LIỆU (OPEN T+1 STRATEGY)
# =============================================================================
def clean_embedding_str(x):
    try:
        if isinstance(x, str):
            x_clean = x.replace('[', '').replace(']', '').replace('\n', ' ').strip()
            arr = np.fromstring(x_clean, sep=' ', dtype=np.float32)
            if arr.shape[0] == EMBEDDING_DIM: return arr
            elif arr.shape[0] > EMBEDDING_DIM: return arr[:EMBEDDING_DIM]
            else: return np.pad(arr, (0, EMBEDDING_DIM - arr.shape[0]))
        elif isinstance(x, (list, np.ndarray)): return np.array(x, dtype=np.float32)
        return np.zeros(EMBEDDING_DIM, dtype=np.float32)
    except: return np.zeros(EMBEDDING_DIM, dtype=np.float32)

def process_dataframe_csv(df_path):
    print(f"Processing: {df_path}")
    if not os.path.exists(df_path):
        print(f"❌ File not found: {df_path}")
        return pd.DataFrame()

    df = pd.read_csv(df_path)
    col_map = {}
    for col in df.columns:
        c_lower = col.lower()
        if 'date' in c_lower: col_map[col] = 'date'
        elif 'ticker' in c_lower or 'tic' in c_lower: col_map[col] = 'tic'
        elif 'embedding' in c_lower: col_map[col] = 'news_embedding'
        elif col in INDICATORS + ['open', 'high', 'low', 'close', 'volume', 'turbulence']: col_map[col] = col

    df.rename(columns=col_map, inplace=True)
    df['date'] = pd.to_datetime(df['date'], dayfirst=False, errors='coerce')
    df = df.dropna(subset=['date']).sort_values(['date', 'tic']).reset_index(drop=True)

    if 'news_embedding' in df.columns:
        df['news_embedding'] = df['news_embedding'].apply(clean_embedding_str)
    else:
        df['news_embedding'] = [np.zeros(EMBEDDING_DIM, dtype=np.float32)] * len(df)

    all_dates = df['date'].unique()
    all_tickers = df['tic'].unique()
    multi_index = pd.MultiIndex.from_product([all_dates, all_tickers], names=['date', 'tic'])
    df = pd.merge(pd.DataFrame(index=multi_index).reset_index(), df, on=['date', 'tic'], how='left')
    df = df.sort_values(['tic', 'date'])

    num_cols = ['open', 'high', 'low', 'close', 'volume'] + INDICATORS + ['turbulence']
    target = [c for c in num_cols if c in df.columns]
    df[target] = df.groupby('tic')[target].transform(lambda x: x.ffill().bfill().fillna(0))

    null_mask = df['news_embedding'].isnull()
    if null_mask.any():
        df.loc[null_mask, 'news_embedding'] = df.loc[null_mask, 'news_embedding'].apply(
            lambda x: np.zeros(EMBEDDING_DIM, dtype=np.float32)
        )
    
    # --- STRATEGY OPEN T+1 ---
    print("⚡ Applying Strategy: Decision at T -> Execution at Open T+1")
    df['close'] = df.groupby('tic')['open'].shift(-1)
    df = df.dropna(subset=['close'])
    
    return df

print("\n--- Loading & Shifting Data ---")
raw_train = process_dataframe_csv(TRAIN_PATH)
raw_test = process_dataframe_csv(TEST_PATH)

if raw_train.empty or raw_test.empty:
    print("❌ Error loading data.")
    trade = pd.DataFrame()
else:
    full_data = pd.concat([raw_train, raw_test]).sort_values(['date', 'tic']).reset_index(drop=True)
    original_test_start = raw_test['date'].min()
    new_split_date = original_test_start - pd.Timedelta(days=0)
    trade = full_data[full_data['date'] >= new_split_date].copy()
    
    trade_dates = sorted(trade['date'].unique())
    trade['day_index'] = trade['date'].map({d: i for i, d in enumerate(trade_dates)})
    trade_env_df = trade.set_index('day_index')
    stock_dimension = trade['tic'].nunique()
    all_tickers = sorted(trade['tic'].unique())

    def get_daily_mean(df):
        return df.groupby('day_index')['news_embedding'].apply(lambda x: np.mean(np.vstack(x.values), axis=0))

    trade_daily = get_daily_mean(trade)
    TRADE_EMBEDDINGS = {}
    TRADE_EMBEDDINGS['Basic'] = trade_daily.to_dict()
    TRADE_EMBEDDINGS['Rolling30'] = {i: r.values.astype(np.float32) for i, r in pd.DataFrame(trade_daily.tolist()).rolling(30, min_periods=1).mean().iterrows()}
    TRADE_EMBEDDINGS['Decay30'] = {i: r.values.astype(np.float32) for i, r in pd.DataFrame(trade_daily.tolist()).ewm(span=30, adjust=False).mean().iterrows()}

# =============================================================================
# BƯỚC 3: ENVIRONMENT
# =============================================================================
class StockTradingEnvEmbeddingAndReward(StockTradingEnv):
    def __init__(self, df, embedding_dict, **kwargs):
        self.embedding_dict = embedding_dict
        self.embedding_dim = EMBEDDING_DIM
        super().__init__(df=df, **kwargs)
        self.action_space = gym.spaces.Box(low=-1, high=1, shape=(self.action_space.shape[0],), dtype=np.float32)
        self.asset_history = [self.initial_amount]

    def _get_daily_embedding(self):
        return self.embedding_dict.get(self.day, np.zeros(self.embedding_dim, dtype=np.float32))

    def reset(self, *, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        emb = self._get_daily_embedding()
        full_obs = np.concatenate((np.array(obs, dtype=np.float32).flatten(), emb))
        self.state = full_obs
        
        fin_dim = 1 + 2 * self.stock_dim + len(INDICATORS) * self.stock_dim
        current_fin_state = self.state[:fin_dim]
        ia = current_fin_state[0] + sum(np.array(current_fin_state[1:(self.stock_dim+1)]) * 
                                         np.array(current_fin_state[(self.stock_dim+1):(self.stock_dim*2+1)]))
        self.asset_history = [ia]
        return full_obs, info

    def step(self, actions):
        fin_dim = 1 + 2 * self.stock_dim + len(INDICATORS) * self.stock_dim
        next_state_fin, reward, done, truncated, info = super().step(actions)
        
        emb = self._get_daily_embedding()
        full_next_state = np.concatenate((np.array(next_state_fin, dtype=np.float32).flatten(), emb))
        self.state = full_next_state
        
        current_fin_state_new = self.state[:fin_dim]
        end_asset = current_fin_state_new[0] + sum(np.array(current_fin_state_new[1:(self.stock_dim+1)]) * np.array(current_fin_state_new[(self.stock_dim+1):(self.stock_dim*2+1)]))
        self.asset_history.append(end_asset)
        
        return full_next_state, reward, done, truncated, info

class EnhancedStockTradingEnvLSTM(StockTradingEnvEmbeddingAndReward):
    def __init__(self, df, window_size, embedding_dict, **kwargs):
        self.window_size = window_size
        n_stocks = kwargs.get('stock_dim', 30)
        n_inds = len(kwargs.get('tech_indicator_list', []))
        self.single_day_dim = 1 + 2 * n_stocks + n_inds * n_stocks + EMBEDDING_DIM
        total_state_space = self.window_size * self.single_day_dim
        kwargs['state_space'] = total_state_space
        super().__init__(df=df, embedding_dict=embedding_dict, **kwargs)
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(total_state_space,), dtype=np.float32)
        self.state_memory_window = deque(maxlen=self.window_size)

    def reset(self, *, seed=None, options=None):
        self.state_memory_window.clear()
        s, _ = super().reset(seed=seed, options=options)
        for _ in range(self.window_size): self.state_memory_window.append(np.array(s, dtype=np.float32))
        return np.concatenate(list(self.state_memory_window)), {}

    def step(self, actions):
        next_s, r, d, t, i = super().step(actions)
        self.state_memory_window.append(np.array(next_s, dtype=np.float32))
        return np.concatenate(list(self.state_memory_window)), r, d, t, i

if not trade.empty:
    base_env_kwargs = {
        "hmax": 200, "initial_amount": 1000000, "num_stock_shares": [0]*stock_dimension,
        "buy_cost_pct": [0.001]*stock_dimension, "sell_cost_pct": [0.001]*stock_dimension,
        "stock_dim": stock_dimension, "tech_indicator_list": INDICATORS, "action_space": stock_dimension, "reward_scaling": 1e-4
    }

# =============================================================================
# BƯỚC 4: ENSEMBLE METHODS (UPDATED: MODEL TRACKING)
# =============================================================================
class EnsembleMethods:
    def __init__(self, models, model_names, stock_dim, dynamic_enabled=False, 
                 short_lookback_days=63, top_k_select=2,
                 long_lookback_months=6, top_n_pool=10):
        
        self.models = models
        self.model_names = model_names # Lưu tên model để tracking
        self.n_models = len(models)
        self.stock_dim = stock_dim
        self.prev_action = None
        self.dynamic_enabled = dynamic_enabled
        
        self.short_lookback_days = short_lookback_days
        self.top_k_select = min(top_k_select, self.n_models)
        self.long_lookback_days = long_lookback_months * 21 
        self.top_n_pool = min(top_n_pool, self.n_models)
        
        max_history = max(self.short_lookback_days, self.long_lookback_days)
        self.model_returns_history = [deque(maxlen=max_history) for _ in range(self.n_models)]
        self.step_counter = 0
        self.current_pool_indices = list(range(self.n_models)) 
        
        # Tracking: Lưu list các index model được chọn ở mỗi bước
        self.selected_indices_log = [] 

    def update_pool_layer_1(self):
        if len(self.model_returns_history[0]) < self.long_lookback_days:
            self.current_pool_indices = list(range(self.n_models))
            return

        scores = []
        valid_indices = []
        for i in range(self.n_models):
            rets = np.array(self.model_returns_history[i])
            rets_long = rets[-self.long_lookback_days:]
            if len(rets_long) < 10: continue
            
            mean_ret = np.mean(rets_long)
            neg_rets = rets_long[rets_long < 0]
            downside_std = np.sqrt(np.mean(neg_rets**2)) if len(neg_rets) > 0 else 1e-9
            sortino = mean_ret / downside_std
            scores.append(sortino)
            valid_indices.append(i)
        
        if not scores:
            self.current_pool_indices = list(range(self.n_models))
            return

        top_indices_local = np.argsort(scores)[-self.top_n_pool:]
        self.current_pool_indices = [valid_indices[i] for i in top_indices_local]

    def select_models_layer_2(self):
        if not self.dynamic_enabled: 
            return list(range(self.n_models))

        scores_map = {} 
        for idx in self.current_pool_indices:
            rets = np.array(self.model_returns_history[idx])
            if len(rets) < self.short_lookback_days: rets_short = rets
            else: rets_short = rets[-self.short_lookback_days:]
            
            if len(rets_short) < 5: 
                scores_map[idx] = -999.0
                continue
            
            mean_ret = np.mean(rets_short)
            neg_rets = rets_short[rets_short < 0]
            downside_std = np.sqrt(np.mean(neg_rets**2)) if len(neg_rets) > 0 else 1e-9
            sortino = mean_ret / downside_std
            scores_map[idx] = sortino
        
        sorted_by_score = sorted(scores_map.items(), key=lambda x: x[1], reverse=True)
        k_final = min(self.top_k_select, len(sorted_by_score))
        top_indices = [x[0] for x in sorted_by_score[:k_final]]
        return top_indices

    def predict(self, obs, lstm_states_list, method_id, prev_action=None, 
                current_prices=None, prev_prices=None): 
        
        # 1. Thu thập Action
        current_actions = []
        new_states = []
        for i, model in enumerate(self.models):
            is_deterministic = (method_id != 3) 
            # Quan trọng: deterministic=True giúp loại bỏ ngẫu nhiên trong policy
            a, s = model.predict(obs, state=lstm_states_list[i], deterministic=is_deterministic)
            if a.ndim > 1: a = a.flatten()
            current_actions.append(a)
            new_states.append(s)
        action_stack = np.array(current_actions)

        # 2. Update Returns
        if current_prices is not None and prev_prices is not None:
            price_change_pct = (current_prices - prev_prices) / (prev_prices + 1e-9)
            price_change_pct = np.clip(price_change_pct, -0.2, 0.2)
            for i in range(self.n_models):
                proxy_return = np.sum(action_stack[i] * price_change_pct)
                self.model_returns_history[i].append(proxy_return)
        
        # 3. Update Lớp 1
        if self.step_counter > 0 and (self.step_counter % self.long_lookback_days == 0):
            self.update_pool_layer_1()
        self.step_counter += 1

        # 4. Aggregation
        final_action = None
        selected_indices = []

        if method_id == 5: 
            selected_indices = self.select_models_layer_2()
            if len(selected_indices) > 0:
                final_action = np.mean(action_stack[selected_indices], axis=0)
            else:
                if len(self.current_pool_indices) > 0:
                    selected_indices = self.current_pool_indices
                    final_action = np.mean(action_stack[self.current_pool_indices], axis=0)
                else:
                    selected_indices = list(range(self.n_models))
                    final_action = np.mean(action_stack, axis=0)
        else:
            selected_indices = list(range(self.n_models))
            final_action = np.mean(action_stack, axis=0)
        
        # Log lại các model đã chọn cho bước này
        self.selected_indices_log.append(selected_indices)
            
        return final_action, new_states

# =============================================================================
# BƯỚC 5: METRICS
# =============================================================================
def calculate_metrics(account_values, initial=1000000):
    df = pd.DataFrame(account_values, columns=['val'])
    df['ret'] = df['val'].pct_change().fillna(0)
    final = df['val'].iloc[-1]
    total_ret = ((final - initial)/initial)*100
    std_dev = df['ret'].std()
    sharpe = (df['ret'].mean()/std_dev*np.sqrt(252)) if std_dev!=0 else 0
    neg_ret = df['ret'][df['ret']<0]
    sortino = (df['ret'].mean()/neg_ret.std()*np.sqrt(252)) if len(neg_ret)>0 and neg_ret.std()!=0 else 0
    cum = (1+df['ret']).cumprod()
    dd = (cum - cum.cummax())/cum.cummax()
    max_dd = dd.min()*100
    return {'final_value': final, 'return': total_ret, 'sharpe': sharpe, 'max_dd': max_dd, 'sortino': sortino}

# =============================================================================
# BƯỚC 6: BACKTEST FUNCTIONS (UPDATED: RETURN LOG)
# =============================================================================
def run_backtest_ensemble(models, model_names, env_df, embedding_dict, env_kwargs, method_id, seed=100, ticker_list=None,
                          l2_lookback_months=3, l2_top_k=2,
                          l1_filter_months=6, l1_top_n=10):
    
    # Reset seed right before env creation
    set_seed(seed)
    
    env = EnhancedStockTradingEnvLSTM(df=env_df, window_size=FIXED_WINDOW_SIZE, embedding_dict=embedding_dict, **env_kwargs)
    obs, _ = env.reset(seed=seed)
    lstm_states = [None]*len(models)
    
    ensemble = EnsembleMethods(
        models, model_names, 
        env_kwargs['stock_dim'],
        dynamic_enabled=True,
        short_lookback_days=l2_lookback_months * 21, 
        top_k_select=l2_top_k,      
        long_lookback_months=l1_filter_months,
        top_n_pool=l1_top_n
    )
    
    alloc_hist = []
    curr_date_idx = 0
    done = False
    prev_prices = None
    
    while not done:
        n_stk = env_kwargs['stock_dim']
        single_dim = env.single_day_dim
        curr_state_vec = obs[-single_dim:] 
        current_prices = curr_state_vec[1 : 1+n_stk]
        
        act, lstm_states = ensemble.predict(
            obs, lstm_states, method_id, 
            prev_action=ensemble.prev_action,
            current_prices=current_prices,
            prev_prices=prev_prices
        )
        prev_prices = current_prices
        if method_id == 4: ensemble.prev_action = act
        
        obs, reward, done, truncated, info = env.step(act)
        
        # Allocation tracking
        curr_vec_next = obs[-single_dim:]
        cash = curr_vec_next[0]
        prices_real = curr_vec_next[1 : 1+n_stk]
        shares_real = curr_vec_next[1+n_stk : 1+2*n_stk]
        vals = shares_real * prices_real
        total = cash + np.sum(vals)
        
        row = {}
        if curr_date_idx < len(trade_dates):
            row['date'] = trade_dates[curr_date_idx]
            if total > 0:
                row['Cash'] = cash/total
                if ticker_list:
                    for i, tic in enumerate(ticker_list): 
                        if i < len(vals): row[tic] = vals[i]/total
            else:
                row['Cash'] = 1.0
                if ticker_list:
                    for tic in ticker_list: row[tic] = 0.0
            alloc_hist.append(row)
        
        curr_date_idx += 1
        if done or truncated: break
            
    alloc_df = pd.DataFrame(alloc_hist)
    if not alloc_df.empty: alloc_df.set_index('date', inplace=True)
    
    # --- BUILD USAGE LOG ---
    usage_data = []
    unique_models_set = set()
    
    # Ghép date với log indices
    # Lưu ý: ensemble.selected_indices_log có độ dài bằng số bước đã chạy
    limit = min(len(trade_dates), len(ensemble.selected_indices_log))
    for i in range(limit):
        indices = ensemble.selected_indices_log[i]
        names = [model_names[idx] for idx in indices]
        unique_models_set.update(names)
        usage_data.append({
            'date': trade_dates[i],
            'selected_models': ", ".join(names),
            'count': len(names)
        })
        
    usage_df = pd.DataFrame(usage_data)
    
    return env.asset_history, alloc_df, usage_df, len(unique_models_set)

# =============================================================================
# BƯỚC 7: LOAD MODELS (CPU FORCED FOR DETERMINISM)
# =============================================================================
all_models = []
all_model_names = []

print("\n--- Scanning & Loading Models (FORCING CPU FOR DETERMINISM) ---")
if TRAINED_MODEL_DIR and os.path.exists(TRAINED_MODEL_DIR):
    model_paths = []
    for root, dirs, files in os.walk(TRAINED_MODEL_DIR):
        if "policy.pth" in files:
            folder_name = os.path.basename(root)
            parent_name = os.path.basename(os.path.dirname(root))
            display_name = folder_name if folder_name != parent_name else parent_name
            model_paths.append({'path': root, 'name': display_name, 'type': 'folder'})
            continue 
        for f in files:
            if f.endswith('.zip') and not f.startswith('.'):
                model_paths.append({'path': os.path.join(root, f), 'name': f.replace('.zip',''), 'type': 'zip'})

    print(f"🎯 Tìm thấy {len(model_paths)} models. Đang tiến hành load...")
    
    for info in model_paths:
        try:
            print(f"➡️ Loading: {info['name']}...", end=" ")
            final_path = info['path']
            temp_zip_path = None
            temp_dir_path = None

            if info['type'] == 'folder' and '/kaggle/input/' in info['path']:
                unique_id = str(hash(info['path']))[-6:]
                temp_dir_path = f"/kaggle/working/temp_{info['name']}_{unique_id}"
                temp_zip_path = temp_dir_path + ".zip"
                if os.path.exists(temp_dir_path): shutil.rmtree(temp_dir_path)
                shutil.copytree(info['path'], temp_dir_path)
                shutil.make_archive(temp_dir_path, 'zip', temp_dir_path)
                final_path = temp_zip_path

            # --- QUAN TRỌNG: device='cpu' để đảm bảo kết quả CPU và GPU giống hệt nhau ---
            model = RecurrentPPO.load(final_path, device='auto') 
            
            if temp_dir_path and os.path.exists(temp_dir_path): shutil.rmtree(temp_dir_path)
            if temp_zip_path and os.path.exists(temp_zip_path): os.remove(temp_zip_path)

            mode = 'Decay30'
            for m in EMBEDDING_MODES: 
                if m.lower() in info['name'].lower(): 
                    mode = m
                    break
                    
            all_models.append({'model': model, 'name': info['name'], 'mode': mode})
            all_model_names.append(info['name'])
            print(f"✅ OK (Mode: {mode})")
        except Exception as e:
            print(f"❌ FAILED: {e}")
            
print(f"--- Tổng cộng đã load thành công: {len(all_models)} model ---")

# =============================================================================
# BƯỚC 8: RUN & EXPORT
# =============================================================================
if not all_models:
    print("❌ Cannot start backtest (No models loaded).")
else:
    print("\n--- Starting Backtest ---")
    models = [m['model'] for m in all_models]
    emb_mode = all_models[0]['mode']
    emb_dict = TRADE_EMBEDDINGS.get(emb_mode, TRADE_EMBEDDINGS['Decay30'])
    
    results = {}
    summary = []

    if ENABLE_DYNAMIC_ENSEMBLE:
        param_grid = product(GRID_L2_LOOKBACK_MONTHS, GRID_L2_TOP_K_MODELS, GRID_L1_FILTER_MONTHS, GRID_L1_TOP_N_POOL)
        
        for l2_lb, l2_k, l1_lb, l1_n in param_grid:
            name = f"5_L1({l1_lb}m_N{l1_n})_L2({l2_lb}m_K{l2_k})"
            
            print(f"Running {name}...", end=" ")
            try:
                hist, alloc_df, usage_df, unique_count = run_backtest_ensemble(
                    models, all_model_names, 
                    trade_env_df, emb_dict, base_env_kwargs, 
                    5, seed=SEED, ticker_list=all_tickers,
                    l2_lookback_months=l2_lb, l2_top_k=l2_k,
                    l1_filter_months=l1_lb, l1_top_n=l1_n
                )
                met = calculate_metrics(hist)
                results[name] = {'hist': hist, 'alloc': alloc_df, 'usage': usage_df, **met}
                
                print(f"Done. Ret: {met['return']:.1f}% | Unique Models Used: {unique_count}")
                
                # Update summary info with model usage count
                summary.append({
                    'Method': name, 
                    'Return (%)': met['return'], 
                    'Sharpe': met['sharpe'], 
                    'Sortino': met['sortino'],
                    'Unique Models Used': unique_count
                })
                
            except Exception as e:
                print(f"Error: {e}")
                import traceback
                traceback.print_exc()

    # --- EXPORT ---
    print("\n--- Exporting Results ---")
    for mname, res in results.items():
        clean_name = "".join(x for x in mname if x.isalnum() or x in "_- ")
        
        # Save History
        hist_vals = res['hist']
        min_len = min(len(trade_dates), len(hist_vals))
        pd.DataFrame({'date': trade_dates[:min_len], 'account_value': hist_vals[:min_len]}).to_csv(f"history_{clean_name}.csv", index=False)
        
        # Save Allocation
        if res['alloc'] is not None: res['alloc'].to_csv(f"allocation_{clean_name}.csv")
            
        # Save Model Usage Log
        if res['usage'] is not None: 
            res['usage'].to_csv(f"usage_log_{clean_name}.csv", index=False)
            
    if summary:
        summary_df = pd.DataFrame(summary).sort_values(by='Sortino', ascending=False)
        summary_df.to_csv("summary_results.csv", index=False)
        print("✅ Saved summary_results.csv")
        print(summary_df.to_string(index=False))
        
    print("\n✅ BACKTEST COMPLETED.")

2026-01-09 09:22:44.703971: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767950564.998842    2195 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767950565.088232    2195 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

✅ Random Seed Set: 100
✓ Found model directory: /kaggle/input/dulieu-hmax200-2022

--- Loading & Shifting Data ---
Processing: /kaggle/input/file-du-lieu-vi-embedding/train2025_sentiment_with_emb.csv
⚡ Applying Strategy: Decision at T -> Execution at Open T+1
Processing: /kaggle/input/test-moi/test2025_sentiment_with_emb2.csv
⚡ Applying Strategy: Decision at T -> Execution at Open T+1

--- Scanning & Loading Models (FORCING CPU FOR DETERMINISM) ---
🎯 Tìm thấy 29 models. Đang tiến hành load...
➡️ Loading: DAPO_PPO_LSTM_Decay30_Seed7_FINAL... ✅ OK (Mode: Decay30)
➡️ Loading: DAPO_PPO_LSTM_Decay30_Seed20_FINAL... ✅ OK (Mode: Decay30)
➡️ Loading: DAPO_PPO_LSTM_Decay30_Seed4_FINAL... ✅ OK (Mode: Decay30)
➡️ Loading: DAPO_PPO_LSTM_Decay30_Seed6_FINAL... ✅ OK (Mode: Decay30)
➡️ Loading: DAPO_PPO_LSTM_Decay30_Seed9_FINAL... ✅ OK (Mode: Decay30)
➡️ Loading: DAPO_PPO_LSTM_Decay30_Seed17_FINAL... ✅ OK (Mode: Decay30)
➡️ Loading: DAPO_PPO_LSTM_Decay30_Seed33_FINAL... ✅ OK (Mode: Decay30)
➡️ Loadin

# BACKTEST OPEN T+1 (THÊM T+2.5)

In [3]:
# =============================================================================
# BACKTEST FIXED: 2-LAYER ENSEMBLE (LOGGING + DETERMINISTIC CPU + T+2.5)
# =============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import shutil
import zipfile
import gymnasium as gym
from sb3_contrib import RecurrentPPO
from collections import deque
from itertools import product
import warnings
import random
import torch

warnings.filterwarnings("ignore")

# --- FinRL Imports ---
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.config import INDICATORS

# =============================================================================
# 1. CẤU HÌNH DETERMINISTIC & PARAMETERS
# =============================================================================
SEED = 100

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"✅ Random Seed Set: {seed}")

set_seed(SEED)

# --- CẤU HÌNH DYNAMIC ENSEMBLE ---
ENABLE_DYNAMIC_ENSEMBLE = True

# --- CẤU HÌNH T+2.5 (MỚI THÊM) ---
HOLD_PERIOD_DAYS = 2      # Cổ phiếu mua xong phải giữ 2.5 ngày
CASH_SETTLEMENT_DAYS = 0   # Tiền bán xong phải chờ 2.5 ngày mới về

# --- GRID SEARCH PARAMETERS ---
# Lớp 2: Chọn model ngắn hạn (Trading)
GRID_L2_LOOKBACK_MONTHS = [1]    
GRID_L2_TOP_K_MODELS    = [1]    

# Lớp 1: Lọc Pool dài hạn (Filter)
GRID_L1_FILTER_MONTHS   = [10]    
GRID_L1_TOP_N_POOL      = [10]    

# --- Cấu hình dữ liệu ---
TRAIN_PATH = '/kaggle/input/file-du-lieu-vi-embedding/train2025_sentiment_with_emb.csv'
TEST_PATH = '/kaggle/input/test-moi/test2025_sentiment_with_emb2.csv'
EMBEDDING_DIM = 1024
FIXED_WINDOW_SIZE = 15
EMBEDDING_MODES = ["Decay30", "Rolling30", "Basic"]

# --- MODEL DIRECTORY ---
TRAINED_MODEL_DIRS = [
    "/kaggle/input/dulieu-hmax200-2022",
    "/kaggle/input/modeltest",
    "./TRAINED_MODEL_DIR"
]

TRAINED_MODEL_DIR = None
for path in TRAINED_MODEL_DIRS:
    if os.path.exists(path):
        TRAINED_MODEL_DIR = path
        print(f"✓ Found model directory: {TRAINED_MODEL_DIR}")
        break

if TRAINED_MODEL_DIR is None:
    print("⚠ CẢNH BÁO: Không tìm thấy thư mục model! Sử dụng mặc định.")
    TRAINED_MODEL_DIR = "/kaggle/working/trained_models"

# =============================================================================
# 2. XỬ LÝ DỮ LIỆU (OPEN T+1 STRATEGY)
# =============================================================================
def clean_embedding_str(x):
    try:
        if isinstance(x, str):
            x_clean = x.replace('[', '').replace(']', '').replace('\n', ' ').strip()
            arr = np.fromstring(x_clean, sep=' ', dtype=np.float32)
            if arr.shape[0] == EMBEDDING_DIM: return arr
            elif arr.shape[0] > EMBEDDING_DIM: return arr[:EMBEDDING_DIM]
            else: return np.pad(arr, (0, EMBEDDING_DIM - arr.shape[0]))
        elif isinstance(x, (list, np.ndarray)): return np.array(x, dtype=np.float32)
        return np.zeros(EMBEDDING_DIM, dtype=np.float32)
    except: return np.zeros(EMBEDDING_DIM, dtype=np.float32)

def process_dataframe_csv(df_path):
    print(f"Processing: {df_path}")
    if not os.path.exists(df_path):
        print(f"❌ File not found: {df_path}")
        return pd.DataFrame()

    df = pd.read_csv(df_path)
    col_map = {}
    for col in df.columns:
        c_lower = col.lower()
        if 'date' in c_lower: col_map[col] = 'date'
        elif 'ticker' in c_lower or 'tic' in c_lower: col_map[col] = 'tic'
        elif 'embedding' in c_lower: col_map[col] = 'news_embedding'
        elif col in INDICATORS + ['open', 'high', 'low', 'close', 'volume', 'turbulence']: col_map[col] = col

    df.rename(columns=col_map, inplace=True)
    df['date'] = pd.to_datetime(df['date'], dayfirst=False, errors='coerce')
    df = df.dropna(subset=['date']).sort_values(['date', 'tic']).reset_index(drop=True)

    if 'news_embedding' in df.columns:
        df['news_embedding'] = df['news_embedding'].apply(clean_embedding_str)
    else:
        df['news_embedding'] = [np.zeros(EMBEDDING_DIM, dtype=np.float32)] * len(df)

    all_dates = df['date'].unique()
    all_tickers = df['tic'].unique()
    multi_index = pd.MultiIndex.from_product([all_dates, all_tickers], names=['date', 'tic'])
    df = pd.merge(pd.DataFrame(index=multi_index).reset_index(), df, on=['date', 'tic'], how='left')
    df = df.sort_values(['tic', 'date'])

    num_cols = ['open', 'high', 'low', 'close', 'volume'] + INDICATORS + ['turbulence']
    target = [c for c in num_cols if c in df.columns]
    df[target] = df.groupby('tic')[target].transform(lambda x: x.ffill().bfill().fillna(0))

    null_mask = df['news_embedding'].isnull()
    if null_mask.any():
        df.loc[null_mask, 'news_embedding'] = df.loc[null_mask, 'news_embedding'].apply(
            lambda x: np.zeros(EMBEDDING_DIM, dtype=np.float32)
        )
    
    # --- STRATEGY OPEN T+1 ---
    print("⚡ Applying Strategy: Decision at T -> Execution at Open T+1")
    df['close'] = df.groupby('tic')['open'].shift(-1)
    df = df.dropna(subset=['close'])
    
    return df

print("\n--- Loading & Shifting Data ---")
raw_train = process_dataframe_csv(TRAIN_PATH)
raw_test = process_dataframe_csv(TEST_PATH)

if raw_train.empty or raw_test.empty:
    print("❌ Error loading data.")
    trade = pd.DataFrame()
else:
    full_data = pd.concat([raw_train, raw_test]).sort_values(['date', 'tic']).reset_index(drop=True)
    original_test_start = raw_test['date'].min()
    new_split_date = original_test_start - pd.Timedelta(days=0)
    trade = full_data[full_data['date'] >= new_split_date].copy()
    
    trade_dates = sorted(trade['date'].unique())
    trade['day_index'] = trade['date'].map({d: i for i, d in enumerate(trade_dates)})
    trade_env_df = trade.set_index('day_index')
    stock_dimension = trade['tic'].nunique()
    all_tickers = sorted(trade['tic'].unique())

    def get_daily_mean(df):
        return df.groupby('day_index')['news_embedding'].apply(lambda x: np.mean(np.vstack(x.values), axis=0))

    trade_daily = get_daily_mean(trade)
    TRADE_EMBEDDINGS = {}
    TRADE_EMBEDDINGS['Basic'] = trade_daily.to_dict()
    TRADE_EMBEDDINGS['Rolling30'] = {i: r.values.astype(np.float32) for i, r in pd.DataFrame(trade_daily.tolist()).rolling(30, min_periods=1).mean().iterrows()}
    TRADE_EMBEDDINGS['Decay30'] = {i: r.values.astype(np.float32) for i, r in pd.DataFrame(trade_daily.tolist()).ewm(span=30, adjust=False).mean().iterrows()}

# =============================================================================
# BƯỚC 3: ENVIRONMENT (UPDATED WITH FIFO LOTS & T+2.5)
# =============================================================================
class StockTradingEnvEmbeddingAndReward(StockTradingEnv):
    def __init__(self, df, embedding_dict, hold_period_days=0, cash_settlement_days=0, **kwargs):
        self.embedding_dict = embedding_dict
        self.embedding_dim = EMBEDDING_DIM
        
        # --- Config T+2.5 ---
        self.hold_period_days = hold_period_days
        self.cash_settlement_days = cash_settlement_days
        
        super().__init__(df=df, **kwargs)
        self.action_space = gym.spaces.Box(low=-1, high=1, shape=(self.action_space.shape[0],), dtype=np.float32)
        self.asset_history = [self.initial_amount]
        
        # --- QUẢN LÝ LÔ CỔ PHIẾU (FIFO) ---
        self.stock_lots = [deque() for _ in range(self.stock_dim)]
        
        # Nếu khởi tạo có sẵn cổ phiếu, coi như lô cũ
        for i in range(self.stock_dim):
            if self.state[1+i] > 0:
                self.stock_lots[i].append([-999.0, self.state[1+i]])

        self.current_day = 0
        self.deferred_sells = {i: [] for i in range(self.stock_dim)}
        self.deferred_cash = []

    def _get_daily_embedding(self):
        return self.embedding_dict.get(self.day, np.zeros(self.embedding_dim, dtype=np.float32))

    def reset(self, *, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        emb = self._get_daily_embedding()
        full_obs = np.concatenate((np.array(obs, dtype=np.float32).flatten(), emb))
        self.state = full_obs
        
        self.asset_history = [self.initial_amount]
        
        # Reset tracking
        self.stock_lots = [deque() for _ in range(self.stock_dim)]
        # Add initial stocks if any
        current_shares = np.array(self.state[1:(self.stock_dim+1)])
        for i in range(self.stock_dim):
            if current_shares[i] > 0:
                self.stock_lots[i].append([-999.0, current_shares[i]])

        self.current_day = 0
        self.deferred_sells = {i: [] for i in range(self.stock_dim)}
        self.deferred_cash = []
        
        return full_obs, info

    def _calculate_total_asset(self):
        fin_dim = 1 + 2 * self.stock_dim + len(INDICATORS) * self.stock_dim
        current_fin_state = self.state[:fin_dim]
        
        cash = current_fin_state[0]
        stocks_val = sum(np.array(current_fin_state[1:(self.stock_dim+1)]) * 
                         np.array(current_fin_state[(self.stock_dim+1):(self.stock_dim*2+1)]))
        
        # Cộng thêm tiền đang chờ về (Deferred Cash)
        pending_cash = sum([amt for _, amt in self.deferred_cash])
        
        return cash + stocks_val + pending_cash

    def step(self, actions):
        # 1.1. Xử lý Tiền về (Cash Settlement)
        cash_received = 0.0
        remaining_cash_queue = []
        for release_day, amount in self.deferred_cash:
            if self.current_day >= release_day:
                cash_received += amount
            else:
                remaining_cash_queue.append((release_day, amount))
        
        self.deferred_cash = remaining_cash_queue
        self.state[0] += cash_received
        
        # 1.2. Xử lý Lệnh bán chờ
        deferred_actions = np.zeros(self.stock_dim)
        for i in range(self.stock_dim):
            ready_sells = [item for item in self.deferred_sells[i] if item[0] <= self.current_day]
            if ready_sells:
                total_deferred_sell_act = sum([item[1] for item in ready_sells])
                deferred_actions[i] = total_deferred_sell_act
                self.deferred_sells[i] = [item for item in self.deferred_sells[i] if item[0] > self.current_day]
        
        # 2. Xử lý Action mới (FIFO check)
        fin_dim = 1 + 2 * self.stock_dim + len(INDICATORS) * self.stock_dim
        current_fin_state = self.state[:fin_dim]
        current_shares = np.array(current_fin_state[1:(self.stock_dim+1)])
        current_prices = np.array(current_fin_state[(self.stock_dim+1):(self.stock_dim*2+1)])
        
        adjusted_actions = actions.copy()
        
        for i in range(self.stock_dim):
            if actions[i] < 0 and current_shares[i] > 0:
                intended_sell_qty = min(abs(actions[i]) * self.hmax, current_shares[i])
                
                # Tính lượng khả dụng
                available_qty = 0.0
                for lot in self.stock_lots[i]:
                    purchase_day, qty = lot
                    if self.current_day - purchase_day >= self.hold_period_days:
                        available_qty += qty
                
                sell_now_qty = min(intended_sell_qty, available_qty)
                defer_qty = intended_sell_qty - sell_now_qty
                
                # Cập nhật action bán ngay
                if sell_now_qty > 0:
                    adjusted_actions[i] = -(sell_now_qty / self.hmax)
                else:
                    adjusted_actions[i] = 0
                
                # Xử lý phần bị khóa
                if defer_qty > 0:
                    remaining_defer = defer_qty
                    for lot in self.stock_lots[i]:
                        purchase_day, qty = lot
                        if self.current_day - purchase_day < self.hold_period_days:
                            take_from_lot = min(remaining_defer, qty)
                            if take_from_lot > 0:
                                unlock_day = purchase_day + self.hold_period_days
                                defer_action_val = -(take_from_lot / self.hmax)
                                self.deferred_sells[i].append((unlock_day, defer_action_val))
                                remaining_defer -= take_from_lot
                            if remaining_defer <= 0.0001: break

        final_actions = adjusted_actions + deferred_actions
        final_actions = np.clip(final_actions, -1, 1)
        
        # 3. Thực thi
        next_state_fin, reward, done, truncated, info = super().step(final_actions)
        
        # 4. Hậu xử lý
        next_shares = np.array(next_state_fin[1:(self.stock_dim+1)])
        
        # 4.1. Update FIFO Lots
        for i in range(self.stock_dim):
            if next_shares[i] < current_shares[i]: # Bán
                shares_sold = current_shares[i] - next_shares[i]
                while shares_sold > 0.0001 and self.stock_lots[i]:
                    oldest_lot = self.stock_lots[i][0]
                    if oldest_lot[1] > shares_sold:
                        oldest_lot[1] -= shares_sold
                        shares_sold = 0
                    else:
                        shares_sold -= oldest_lot[1]
                        self.stock_lots[i].popleft()
            elif next_shares[i] > current_shares[i]: # Mua
                shares_bought = next_shares[i] - current_shares[i]
                self.stock_lots[i].append([self.current_day, shares_bought])

        # 4.2. Cash Settlement
        cash_to_defer = 0.0
        for i in range(self.stock_dim):
            if final_actions[i] < 0:
                shares_sold_actual = current_shares[i] - next_shares[i]
                if shares_sold_actual > 0: 
                    sell_price = current_prices[i]
                    sell_cost = self.sell_cost_pct[i]
                    revenue = sell_price * shares_sold_actual * (1 - sell_cost)
                    cash_to_defer += revenue
        
        if cash_to_defer > 0:
            next_state_fin[0] -= cash_to_defer
            settlement_day = self.current_day + self.cash_settlement_days
            self.deferred_cash.append((settlement_day, cash_to_defer))
        
        self.current_day += 1
        
        emb = self._get_daily_embedding()
        full_next_state = np.concatenate((np.array(next_state_fin, dtype=np.float32).flatten(), emb))
        self.state = full_next_state
        end_asset = self._calculate_total_asset()
        self.asset_history.append(end_asset)
        
        return full_next_state, reward, done, truncated, info

class EnhancedStockTradingEnvLSTM(StockTradingEnvEmbeddingAndReward):
    def __init__(self, df, window_size, embedding_dict, hold_period_days=0, cash_settlement_days=0, **kwargs):
        self.window_size = window_size
        n_stocks = kwargs.get('stock_dim', 30)
        n_inds = len(kwargs.get('tech_indicator_list', []))
        self.single_day_dim = 1 + 2 * n_stocks + n_inds * n_stocks + EMBEDDING_DIM
        total_state_space = self.window_size * self.single_day_dim
        kwargs['state_space'] = total_state_space
        
        # Pass T+2.5 args
        super().__init__(df=df, embedding_dict=embedding_dict, 
                         hold_period_days=hold_period_days,
                         cash_settlement_days=cash_settlement_days,
                         **kwargs)
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(total_state_space,), dtype=np.float32)
        self.state_memory_window = deque(maxlen=self.window_size)

    def reset(self, *, seed=None, options=None):
        self.state_memory_window.clear()
        s, _ = super().reset(seed=seed, options=options)
        for _ in range(self.window_size): self.state_memory_window.append(np.array(s, dtype=np.float32))
        return np.concatenate(list(self.state_memory_window)), {}

    def step(self, actions):
        next_s, r, d, t, i = super().step(actions)
        self.state_memory_window.append(np.array(next_s, dtype=np.float32))
        return np.concatenate(list(self.state_memory_window)), r, d, t, i

if not trade.empty:
    base_env_kwargs = {
        "hmax": 200, "initial_amount": 1000000, "num_stock_shares": [0]*stock_dimension,
        "buy_cost_pct": [0.001]*stock_dimension, "sell_cost_pct": [0.001]*stock_dimension,
        "stock_dim": stock_dimension, "tech_indicator_list": INDICATORS, "action_space": stock_dimension, "reward_scaling": 1e-4
    }

# =============================================================================
# BƯỚC 4: ENSEMBLE METHODS (UPDATED: MODEL TRACKING)
# =============================================================================
class EnsembleMethods:
    def __init__(self, models, model_names, stock_dim, dynamic_enabled=False, 
                 short_lookback_days=63, top_k_select=2,
                 long_lookback_months=6, top_n_pool=10):
        
        self.models = models
        self.model_names = model_names # Lưu tên model để tracking
        self.n_models = len(models)
        self.stock_dim = stock_dim
        self.prev_action = None
        self.dynamic_enabled = dynamic_enabled
        
        self.short_lookback_days = short_lookback_days
        self.top_k_select = min(top_k_select, self.n_models)
        self.long_lookback_days = long_lookback_months * 21 
        self.top_n_pool = min(top_n_pool, self.n_models)
        
        max_history = max(self.short_lookback_days, self.long_lookback_days)
        self.model_returns_history = [deque(maxlen=max_history) for _ in range(self.n_models)]
        self.step_counter = 0
        self.current_pool_indices = list(range(self.n_models)) 
        
        # Tracking: Lưu list các index model được chọn ở mỗi bước
        self.selected_indices_log = [] 

    def update_pool_layer_1(self):
        if len(self.model_returns_history[0]) < self.long_lookback_days:
            self.current_pool_indices = list(range(self.n_models))
            return

        scores = []
        valid_indices = []
        for i in range(self.n_models):
            rets = np.array(self.model_returns_history[i])
            rets_long = rets[-self.long_lookback_days:]
            if len(rets_long) < 10: continue
            
            mean_ret = np.mean(rets_long)
            neg_rets = rets_long[rets_long < 0]
            downside_std = np.sqrt(np.mean(neg_rets**2)) if len(neg_rets) > 0 else 1e-9
            sortino = mean_ret / downside_std
            scores.append(sortino)
            valid_indices.append(i)
        
        if not scores:
            self.current_pool_indices = list(range(self.n_models))
            return

        top_indices_local = np.argsort(scores)[-self.top_n_pool:]
        self.current_pool_indices = [valid_indices[i] for i in top_indices_local]

    def select_models_layer_2(self):
        if not self.dynamic_enabled: 
            return list(range(self.n_models))

        scores_map = {} 
        for idx in self.current_pool_indices:
            rets = np.array(self.model_returns_history[idx])
            if len(rets) < self.short_lookback_days: rets_short = rets
            else: rets_short = rets[-self.short_lookback_days:]
            
            if len(rets_short) < 5: 
                scores_map[idx] = -999.0
                continue
            
            mean_ret = np.mean(rets_short)
            neg_rets = rets_short[rets_short < 0]
            downside_std = np.sqrt(np.mean(neg_rets**2)) if len(neg_rets) > 0 else 1e-9
            sortino = mean_ret / downside_std
            scores_map[idx] = sortino
        
        sorted_by_score = sorted(scores_map.items(), key=lambda x: x[1], reverse=True)
        k_final = min(self.top_k_select, len(sorted_by_score))
        top_indices = [x[0] for x in sorted_by_score[:k_final]]
        return top_indices

    def predict(self, obs, lstm_states_list, method_id, prev_action=None, 
                current_prices=None, prev_prices=None): 
        
        # 1. Thu thập Action
        current_actions = []
        new_states = []
        for i, model in enumerate(self.models):
            is_deterministic = (method_id != 3) 
            # Quan trọng: deterministic=True giúp loại bỏ ngẫu nhiên trong policy
            a, s = model.predict(obs, state=lstm_states_list[i], deterministic=is_deterministic)
            if a.ndim > 1: a = a.flatten()
            current_actions.append(a)
            new_states.append(s)
        action_stack = np.array(current_actions)

        # 2. Update Returns
        if current_prices is not None and prev_prices is not None:
            price_change_pct = (current_prices - prev_prices) / (prev_prices + 1e-9)
            price_change_pct = np.clip(price_change_pct, -0.2, 0.2)
            for i in range(self.n_models):
                proxy_return = np.sum(action_stack[i] * price_change_pct)
                self.model_returns_history[i].append(proxy_return)
        
        # 3. Update Lớp 1
        if self.step_counter > 0 and (self.step_counter % self.long_lookback_days == 0):
            self.update_pool_layer_1()
        self.step_counter += 1

        # 4. Aggregation
        final_action = None
        selected_indices = []

        if method_id == 5: 
            selected_indices = self.select_models_layer_2()
            if len(selected_indices) > 0:
                final_action = np.mean(action_stack[selected_indices], axis=0)
            else:
                if len(self.current_pool_indices) > 0:
                    selected_indices = self.current_pool_indices
                    final_action = np.mean(action_stack[self.current_pool_indices], axis=0)
                else:
                    selected_indices = list(range(self.n_models))
                    final_action = np.mean(action_stack, axis=0)
        else:
            selected_indices = list(range(self.n_models))
            final_action = np.mean(action_stack, axis=0)
        
        # Log lại các model đã chọn cho bước này
        self.selected_indices_log.append(selected_indices)
            
        return final_action, new_states

# =============================================================================
# BƯỚC 5: METRICS
# =============================================================================
def calculate_metrics(account_values, initial=1000000):
    df = pd.DataFrame(account_values, columns=['val'])
    df['ret'] = df['val'].pct_change().fillna(0)
    final = df['val'].iloc[-1]
    total_ret = ((final - initial)/initial)*100
    std_dev = df['ret'].std()
    sharpe = (df['ret'].mean()/std_dev*np.sqrt(252)) if std_dev!=0 else 0
    neg_ret = df['ret'][df['ret']<0]
    sortino = (df['ret'].mean()/neg_ret.std()*np.sqrt(252)) if len(neg_ret)>0 and neg_ret.std()!=0 else 0
    cum = (1+df['ret']).cumprod()
    dd = (cum - cum.cummax())/cum.cummax()
    max_dd = dd.min()*100
    return {'final_value': final, 'return': total_ret, 'sharpe': sharpe, 'max_dd': max_dd, 'sortino': sortino}

# =============================================================================
# BƯỚC 6: BACKTEST FUNCTIONS (UPDATED: RETURN LOG + T+2.5 PARAMS)
# =============================================================================
def run_backtest_ensemble(models, model_names, env_df, embedding_dict, env_kwargs, method_id, seed=100, ticker_list=None,
                          l2_lookback_months=3, l2_top_k=2,
                          l1_filter_months=6, l1_top_n=10,
                          hold_period_days=0, cash_settlement_days=0): # Added args
    
    # Reset seed right before env creation
    set_seed(seed)
    
    # Init env with T+2.5 params
    env = EnhancedStockTradingEnvLSTM(
        df=env_df, 
        window_size=FIXED_WINDOW_SIZE, 
        embedding_dict=embedding_dict, 
        hold_period_days=hold_period_days,
        cash_settlement_days=cash_settlement_days,
        **env_kwargs
    )
    
    obs, _ = env.reset(seed=seed)
    lstm_states = [None]*len(models)
    
    ensemble = EnsembleMethods(
        models, model_names, 
        env_kwargs['stock_dim'],
        dynamic_enabled=True,
        short_lookback_days=l2_lookback_months * 21, 
        top_k_select=l2_top_k,      
        long_lookback_months=l1_filter_months,
        top_n_pool=l1_top_n
    )
    
    alloc_hist = []
    curr_date_idx = 0
    done = False
    prev_prices = None
    
    while not done:
        n_stk = env_kwargs['stock_dim']
        single_dim = env.single_day_dim
        curr_state_vec = obs[-single_dim:] 
        current_prices = curr_state_vec[1 : 1+n_stk]
        
        act, lstm_states = ensemble.predict(
            obs, lstm_states, method_id, 
            prev_action=ensemble.prev_action,
            current_prices=current_prices,
            prev_prices=prev_prices
        )
        prev_prices = current_prices
        if method_id == 4: ensemble.prev_action = act
        
        obs, reward, done, truncated, info = env.step(act)
        
        # Allocation tracking (Use total asset from env history)
        # Vì env.asset_history đã tính chuẩn pending cash
        total = env.asset_history[-1]
        
        curr_vec_next = obs[-single_dim:]
        cash = curr_vec_next[0]
        prices_real = curr_vec_next[1 : 1+n_stk]
        shares_real = curr_vec_next[1+n_stk : 1+2*n_stk]
        vals = shares_real * prices_real
        # Note: 'total' ở đây lấy từ history env đã xử lý T+2.5
        
        row = {}
        if curr_date_idx < len(trade_dates):
            row['date'] = trade_dates[curr_date_idx]
            if total > 0:
                row['Cash'] = cash/total
                if ticker_list:
                    for i, tic in enumerate(ticker_list): 
                        if i < len(vals): row[tic] = vals[i]/total
            else:
                row['Cash'] = 1.0
                if ticker_list:
                    for tic in ticker_list: row[tic] = 0.0
            alloc_hist.append(row)
        
        curr_date_idx += 1
        if done or truncated: break
            
    alloc_df = pd.DataFrame(alloc_hist)
    if not alloc_df.empty: alloc_df.set_index('date', inplace=True)
    
    # --- BUILD USAGE LOG ---
    usage_data = []
    unique_models_set = set()
    
    limit = min(len(trade_dates), len(ensemble.selected_indices_log))
    for i in range(limit):
        indices = ensemble.selected_indices_log[i]
        names = [model_names[idx] for idx in indices]
        unique_models_set.update(names)
        usage_data.append({
            'date': trade_dates[i],
            'selected_models': ", ".join(names),
            'count': len(names)
        })
        
    usage_df = pd.DataFrame(usage_data)
    
    return env.asset_history, alloc_df, usage_df, len(unique_models_set)

# =============================================================================
# BƯỚC 7: LOAD MODELS (CPU FORCED FOR DETERMINISM)
# =============================================================================
all_models = []
all_model_names = []

print("\n--- Scanning & Loading Models (FORCING CPU FOR DETERMINISM) ---")
if TRAINED_MODEL_DIR and os.path.exists(TRAINED_MODEL_DIR):
    model_paths = []
    for root, dirs, files in os.walk(TRAINED_MODEL_DIR):
        if "policy.pth" in files:
            folder_name = os.path.basename(root)
            parent_name = os.path.basename(os.path.dirname(root))
            display_name = folder_name if folder_name != parent_name else parent_name
            model_paths.append({'path': root, 'name': display_name, 'type': 'folder'})
            continue 
        for f in files:
            if f.endswith('.zip') and not f.startswith('.'):
                model_paths.append({'path': os.path.join(root, f), 'name': f.replace('.zip',''), 'type': 'zip'})

    print(f"🎯 Tìm thấy {len(model_paths)} models. Đang tiến hành load...")
    
    for info in model_paths:
        try:
            print(f"➡️ Loading: {info['name']}...", end=" ")
            final_path = info['path']
            temp_zip_path = None
            temp_dir_path = None

            if info['type'] == 'folder' and '/kaggle/input/' in info['path']:
                unique_id = str(hash(info['path']))[-6:]
                temp_dir_path = f"/kaggle/working/temp_{info['name']}_{unique_id}"
                temp_zip_path = temp_dir_path + ".zip"
                if os.path.exists(temp_dir_path): shutil.rmtree(temp_dir_path)
                shutil.copytree(info['path'], temp_dir_path)
                shutil.make_archive(temp_dir_path, 'zip', temp_dir_path)
                final_path = temp_zip_path

            model = RecurrentPPO.load(final_path, device='auto') 
            
            if temp_dir_path and os.path.exists(temp_dir_path): shutil.rmtree(temp_dir_path)
            if temp_zip_path and os.path.exists(temp_zip_path): os.remove(temp_zip_path)

            mode = 'Decay30'
            for m in EMBEDDING_MODES: 
                if m.lower() in info['name'].lower(): 
                    mode = m
                    break
                    
            all_models.append({'model': model, 'name': info['name'], 'mode': mode})
            all_model_names.append(info['name'])
            print(f"✅ OK (Mode: {mode})")
        except Exception as e:
            print(f"❌ FAILED: {e}")
            
print(f"--- Tổng cộng đã load thành công: {len(all_models)} model ---")

# =============================================================================
# BƯỚC 8: RUN & EXPORT (WITH T+2.5 CONSTANTS)
# =============================================================================
if not all_models:
    print("❌ Cannot start backtest (No models loaded).")
else:
    print(f"\n--- Starting Backtest (Hold Period: {HOLD_PERIOD_DAYS}d | Settlement: {CASH_SETTLEMENT_DAYS}d) ---")
    models = [m['model'] for m in all_models]
    emb_mode = all_models[0]['mode']
    emb_dict = TRADE_EMBEDDINGS.get(emb_mode, TRADE_EMBEDDINGS['Decay30'])
    
    results = {}
    summary = []

    if ENABLE_DYNAMIC_ENSEMBLE:
        param_grid = product(GRID_L2_LOOKBACK_MONTHS, GRID_L2_TOP_K_MODELS, GRID_L1_FILTER_MONTHS, GRID_L1_TOP_N_POOL)
        
        for l2_lb, l2_k, l1_lb, l1_n in param_grid:
            name = f"5_L1({l1_lb}m_N{l1_n})_L2({l2_lb}m_K{l2_k})"
            
            print(f"Running {name}...", end=" ")
            try:
                hist, alloc_df, usage_df, unique_count = run_backtest_ensemble(
                    models, all_model_names, 
                    trade_env_df, emb_dict, base_env_kwargs, 
                    5, seed=SEED, ticker_list=all_tickers,
                    l2_lookback_months=l2_lb, l2_top_k=l2_k,
                    l1_filter_months=l1_lb, l1_top_n=l1_n,
                    hold_period_days=HOLD_PERIOD_DAYS,        # Pass Constant
                    cash_settlement_days=CASH_SETTLEMENT_DAYS # Pass Constant
                )
                met = calculate_metrics(hist)
                results[name] = {'hist': hist, 'alloc': alloc_df, 'usage': usage_df, **met}
                
                print(f"Done. Ret: {met['return']:.1f}% | Unique Models Used: {unique_count}")
                
                summary.append({
                    'Method': name, 
                    'Return (%)': met['return'], 
                    'Sharpe': met['sharpe'], 
                    'Sortino': met['sortino'],
                    'Unique Models Used': unique_count
                })
                
            except Exception as e:
                print(f"Error: {e}")
                import traceback
                traceback.print_exc()

    # --- EXPORT ---
    print("\n--- Exporting Results ---")
    for mname, res in results.items():
        clean_name = "".join(x for x in mname if x.isalnum() or x in "_- ")
        
        hist_vals = res['hist']
        min_len = min(len(trade_dates), len(hist_vals))
        pd.DataFrame({'date': trade_dates[:min_len], 'account_value': hist_vals[:min_len]}).to_csv(f"history_{clean_name}.csv", index=False)
        
        if res['alloc'] is not None: res['alloc'].to_csv(f"allocation_{clean_name}.csv")
        if res['usage'] is not None: res['usage'].to_csv(f"usage_log_{clean_name}.csv", index=False)
            
    if summary:
        summary_df = pd.DataFrame(summary).sort_values(by='Sortino', ascending=False)
        summary_df.to_csv("summary_results.csv", index=False)
        print("✅ Saved summary_results.csv")
        print(summary_df.to_string(index=False))
        
    print("\n✅ BACKTEST COMPLETED.")

✅ Random Seed Set: 100
✓ Found model directory: /kaggle/input/dulieu-hmax200-2022

--- Loading & Shifting Data ---
Processing: /kaggle/input/file-du-lieu-vi-embedding/train2025_sentiment_with_emb.csv
⚡ Applying Strategy: Decision at T -> Execution at Open T+1
Processing: /kaggle/input/test-moi/test2025_sentiment_with_emb2.csv
⚡ Applying Strategy: Decision at T -> Execution at Open T+1

--- Scanning & Loading Models (FORCING CPU FOR DETERMINISM) ---
🎯 Tìm thấy 29 models. Đang tiến hành load...
➡️ Loading: DAPO_PPO_LSTM_Decay30_Seed7_FINAL... ✅ OK (Mode: Decay30)
➡️ Loading: DAPO_PPO_LSTM_Decay30_Seed20_FINAL... ✅ OK (Mode: Decay30)
➡️ Loading: DAPO_PPO_LSTM_Decay30_Seed4_FINAL... ✅ OK (Mode: Decay30)
➡️ Loading: DAPO_PPO_LSTM_Decay30_Seed6_FINAL... ✅ OK (Mode: Decay30)
➡️ Loading: DAPO_PPO_LSTM_Decay30_Seed9_FINAL... ✅ OK (Mode: Decay30)
➡️ Loading: DAPO_PPO_LSTM_Decay30_Seed17_FINAL... ✅ OK (Mode: Decay30)
➡️ Loading: DAPO_PPO_LSTM_Decay30_Seed33_FINAL... ✅ OK (Mode: Decay30)
➡️ Loadin

# VẼ HÌNH 

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import numpy as np
import os
from scipy import stats

# =============================================================================
# 0. SỬA LỖI FONT (QUAN TRỌNG)
# =============================================================================
# Thiết lập font fallback để tránh lỗi 'Arial not found'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Liberation Sans', 'Tahoma', 'Geneva', 'Arial']
plt.rcParams['axes.unicode_minus'] = False # Sửa lỗi hiển thị dấu trừ

# =============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# =============================================================================
METHOD_NAME = "5_L110m_N10_L21m_K1_opent1" 
INITIAL_CAPITAL = 1_000_000
CASH_THRESHOLD = 0.0 

# Đường dẫn file kết quả
ALLOCATION_FILE = f"/kaggle/input/model-open-t-1/allocation_{METHOD_NAME}.csv"
HISTORY_FILE = f"/kaggle/input/model-open-t-1/history_{METHOD_NAME}.csv"

# Đường dẫn dữ liệu Test và VNIndex
TEST_DATA_PATH = '/kaggle/input/test-moi/test2025_sentiment_with_emb2.csv'
VNINDEX_PATH = '/kaggle/input/vnindex-chiso/vnindex_chiso.csv'

print(f"--- ĐANG CHUẨN BỊ DỮ LIỆU ĐỂ VẼ HÌNH ---")

# =============================================================================
# 2. HÀM HỖ TRỢ
# =============================================================================
def calculate_drawdown(series):
    roll_max = series.cummax()
    drawdown = (series - roll_max) / roll_max
    return drawdown * 100

def rebase_series(series, start_val=INITIAL_CAPITAL):
    if len(series) == 0: return series
    return (series / series.iloc[0]) * start_val

def load_and_process_vnindex(path, start_date, end_date):
    if not os.path.exists(path):
        print(f"Không tìm thấy file VNIndex tại: {path}")
        return None
    try:
        df = pd.read_csv(path)
        df.columns = [c.lower().strip() for c in df.columns]
        date_col = next((c for c in df.columns if c in ['date', 'ngay', 'time']), None)
        price_col = next((c for c in df.columns if c in ['close', 'price', 'dong', 'adj close']), None)
        
        if not date_col or not price_col: return None
        
        # Parse ngày tháng (format M/D/Y)
        df[date_col] = pd.to_datetime(df[date_col], dayfirst=False, errors='coerce')
        df = df.dropna(subset=[date_col]).sort_values(date_col).set_index(date_col)
        
        mask = (df.index >= start_date) & (df.index <= end_date)
        df_filtered = df.loc[mask, [price_col]].rename(columns={price_col: 'vnindex'})
        return df_filtered
    except Exception as e:
        print(f"Lỗi VNIndex: {e}")
        return None

# =============================================================================
# 3. HÀM TÍNH TOÁN CHỈ SỐ TÀI CHÍNH
# =============================================================================
def calculate_financial_metrics(series, name="Strategy"):
    if series is None or len(series) == 0: return None
    returns = series.pct_change().dropna()
    trading_days = len(returns)
    years = trading_days / 252
    
    total_return = (series.iloc[-1] / series.iloc[0] - 1) * 100
    annualized_return = ((series.iloc[-1] / series.iloc[0]) ** (1/years) - 1) * 100
    daily_std = returns.std()
    annualized_vol = daily_std * np.sqrt(252) * 100
    
    mean_daily_return = returns.mean()
    sharpe = (mean_daily_return / daily_std) * np.sqrt(252) if daily_std != 0 else 0
    
    downside_returns = returns[returns < 0]
    downside_std = downside_returns.std() if len(downside_returns) > 0 else 1e-9
    sortino = (mean_daily_return / downside_std) * np.sqrt(252) if downside_std != 0 else 0
    
    drawdown = calculate_drawdown(series)
    max_dd = drawdown.min()
    calmar = abs(annualized_return / max_dd) if max_dd != 0 else 0
    
    return {
        'Strategy': name,
        'Total Return (%)': round(total_return, 2),
        'Annualized Return (%)': round(annualized_return, 2),
        'Annualized Volatility (%)': round(annualized_vol, 2),
        'Sharpe Ratio': round(sharpe, 3),
        'Sortino Ratio': round(sortino, 3),
        'Calmar Ratio': round(calmar, 3),
        'Max Drawdown (%)': round(max_dd, 2),
        'Trading Days': trading_days
    }

# =============================================================================
# 4. XỬ LÝ DỮ LIỆU
# =============================================================================
try:
    if not os.path.exists(HISTORY_FILE): raise FileNotFoundError("Thiếu file kết quả RL.")
        
    df_rl = pd.read_csv(HISTORY_FILE)
    df_alloc = pd.read_csv(ALLOCATION_FILE)
    
    val_col = 'val' if 'val' in df_rl.columns else 'account_value'
    df_rl.rename(columns={val_col: 'RL_Value', 'date': 'date'}, inplace=True)
    df_rl['date'] = pd.to_datetime(df_rl['date'])
    df_rl.set_index('date', inplace=True)
    
    df_alloc['date'] = pd.to_datetime(df_alloc['date'])
    df_alloc.set_index('date', inplace=True)
    df_alloc['Cash'] = df_alloc['Cash'].clip(0, 1)
    
    # Xác định ngày bắt đầu
    fully_invested_mask = df_alloc['Cash'] <= CASH_THRESHOLD
    start_date = df_alloc[fully_invested_mask].index[0] if fully_invested_mask.any() else df_rl.index[0]
    
    df_rl = df_rl[df_rl.index >= start_date]
    df_alloc = df_alloc[df_alloc.index >= start_date]
    df_rl['RL_Value'] = rebase_series(df_rl['RL_Value'])
    end_date = df_rl.index[-1]
    
    # Load Benchmark Stock Data
    raw_df = pd.read_csv(TEST_DATA_PATH)
    raw_df.columns = [c.lower() for c in raw_df.columns]
    raw_df.rename(columns={'ticker': 'tic'}, inplace=True)
    raw_df['date'] = pd.to_datetime(raw_df['date'])
    df_close = raw_df.pivot(index='date', columns='tic', values='close').fillna(method='ffill').fillna(method='bfill')
    df_close = df_close[(df_close.index >= start_date) & (df_close.index <= end_date)]
    
    # Buy & Hold
    if not df_close.empty:
        ew_hold_value = df_close.mul(INITIAL_CAPITAL / len(df_close.columns) / df_close.iloc[0], axis=1).sum(axis=1)
    else:
        ew_hold_value = pd.Series()

    # Equal Weight Rebalance
    ew_rebalance_value = pd.Series(index=df_close.index, dtype=float)
    if not df_close.empty:
        current_value = INITIAL_CAPITAL
        last_month = None
        shares = None
        for date in df_close.index:
            if (date.year, date.month) != last_month:
                shares = (current_value / len(df_close.columns)) / df_close.loc[date]
                last_month = (date.year, date.month)
            current_value = (shares * df_close.loc[date]).sum()
            ew_rebalance_value.loc[date] = current_value

    # VN-Index
    df_vnindex = load_and_process_vnindex(VNINDEX_PATH, start_date, end_date)
    if df_vnindex is not None: df_vnindex['vnindex_scaled'] = rebase_series(df_vnindex['vnindex'])

except Exception as e:
    print(f"LỖI DỮ LIỆU: {e}")
    df_rl = pd.DataFrame()

# =============================================================================
# 5. PHÂN TÍCH GIAO DỊCH
# =============================================================================
def analyze_transactions(df_alloc, df_rl):
    stock_cols = [c for c in df_alloc.columns if c != 'Cash']
    df_changes = df_alloc[stock_cols].diff()
    monthly_summary = []
    
    for date in df_alloc.index[1:]:
        nav = df_rl.loc[date, 'RL_Value'] if date in df_rl.index else INITIAL_CAPITAL
        buy_val = 0
        sell_val = 0
        
        for stock in stock_cols:
            w_change = df_changes.loc[date, stock]
            if pd.notna(w_change) and abs(w_change) > 0.001:
                val = abs(w_change) * nav
                if w_change > 0: buy_val += val
                else: sell_val += val
                
        if buy_val > 0 or sell_val > 0:
            monthly_summary.append({
                'date': date,
                'year_month': date.strftime('%Y-%m'),
                'total_buy_value': buy_val,
                'total_sell_value': sell_val,
                'buy_turnover_pct': (buy_val / nav) * 100,
                'sell_turnover_pct': (sell_val / nav) * 100,
                'nav': nav
            })
            
    df_daily = pd.DataFrame(monthly_summary)
    if df_daily.empty: return pd.DataFrame()
    
    monthly_stats = df_daily.groupby('year_month').agg({
        'buy_turnover_pct': 'sum',
        'sell_turnover_pct': 'sum',
        'nav': 'last'
    }).reset_index()
    
    return monthly_stats

df_monthly_trades = pd.DataFrame()
if not df_alloc.empty:
    df_monthly_trades = analyze_transactions(df_alloc, df_rl)

# =============================================================================
# 6. XUẤT CHỈ SỐ TÀI CHÍNH
# =============================================================================
if not df_rl.empty:
    metrics = []
    metrics.append(calculate_financial_metrics(df_rl['RL_Value'], "RL Agent"))
    if not ew_hold_value.empty: metrics.append(calculate_financial_metrics(ew_hold_value, "Buy & Hold"))
    if not ew_rebalance_value.empty: metrics.append(calculate_financial_metrics(ew_rebalance_value, "EW Rebalance"))
    if df_vnindex is not None: metrics.append(calculate_financial_metrics(df_vnindex['vnindex_scaled'], "VN-Index"))
    
    pd.DataFrame([m for m in metrics if m]).to_excel(f"financial_metrics_{METHOD_NAME}.xlsx", index=False)
    print("Đã xuất file metrics.")

# =============================================================================
# 7. VẼ BIỂU ĐỒ (HÌNH 1-6 VÀ HÌNH 5.2 MỚI)
# =============================================================================
if not df_rl.empty:
    print("\n--- ĐANG VẼ BIỂU ĐỒ ---")
    
    # --- HÌNH 1: WEALTH CHART ---
    fig1, ax1 = plt.subplots(figsize=(16, 8))
    ax1.plot(df_rl.index, df_rl['RL_Value'], label='RL Agent', color='#0066CC', lw=2.5)
    if not ew_hold_value.empty: ax1.plot(ew_hold_value.index, ew_hold_value, label='Buy & Hold', color='#FF6B35', alpha=0.9)
    if not ew_rebalance_value.empty: ax1.plot(ew_rebalance_value.index, ew_rebalance_value, label='EW Rebalance', color='#28A745', linestyle='--')
    if df_vnindex is not None: ax1.plot(df_vnindex.index, df_vnindex['vnindex_scaled'], label='VN-Index', color='#8B008B', linestyle=':')
    
    ax1.set_title(f"Hiệu suất đầu tư: {METHOD_NAME}", fontsize=14, fontweight='bold')
    ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M'))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    plt.savefig(f"backtest_{METHOD_NAME}_wealth.png", dpi=100)
    plt.show()
    
    # --- CÁC HÌNH DRAWDOWN (2,3,4,5) ---
    def plot_drawdown(series, title, color, filename):
        if series is None or len(series) == 0: return
        fig, ax = plt.subplots(figsize=(16, 4))
        dd = calculate_drawdown(series)
        ax.plot(dd.index, dd, color=color, lw=1.5)
        ax.fill_between(dd.index, 0, dd, color=color, alpha=0.3)
        ax.set_title(f"{title} Drawdown", fontweight='bold')
        ax.set_ylabel("Drawdown (%)")
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
        plt.savefig(filename, dpi=100, bbox_inches='tight')
        plt.show()
        
    plot_drawdown(df_rl['RL_Value'], "RL Agent", '#0066CC', f"backtest_{METHOD_NAME}_dd_rl.png")
    plot_drawdown(ew_hold_value, "Buy & Hold", '#FF6B35', f"backtest_{METHOD_NAME}_dd_hold.png")
    plot_drawdown(ew_rebalance_value, "EW Rebalance", '#28A745', f"backtest_{METHOD_NAME}_dd_rebalance.png")
    if df_vnindex is not None: plot_drawdown(df_vnindex['vnindex_scaled'], "VN-Index", '#8B008B', f"backtest_{METHOD_NAME}_dd_vnindex.png")

    # --- HÌNH 6: ALLOCATION ---
    fig6, ax6 = plt.subplots(figsize=(16, 8))
    stock_cols = [c for c in df_alloc.columns if c != 'Cash']
    mean_weights = df_alloc[stock_cols].mean().sort_values(ascending=False)
    sorted_stocks = mean_weights.index.tolist()
    labels = ['Cash'] + sorted_stocks
    y_stack = [df_alloc['Cash'].values] + [df_alloc[col].values for col in sorted_stocks]
    
    colors = ['#D3D3D3'] + [plt.cm.tab20(i % 20) for i in range(len(sorted_stocks))]
    ax6.stackplot(df_alloc.index, y_stack, labels=labels, colors=colors, alpha=0.9)
    ax6.set_title("Phân bổ tài sản", fontsize=14, fontweight='bold')
    ax6.set_ylim(0, 1.0)
    ax6.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
    
    handles, lbls = ax6.get_legend_handles_labels()
    ax6.legend(handles[:10], lbls[:10], loc='upper left', bbox_to_anchor=(1.01, 1), title="Top Holdings")
    plt.tight_layout()
    plt.savefig(f"backtest_{METHOD_NAME}_allocation.png", dpi=100)
    plt.show()

    # =========================================================================
    # HÌNH 5.2 (THAY THẾ HÌNH 7): 2 PHIÊN BẢN
    # =========================================================================
    if not df_monthly_trades.empty:
        print("\n--- ĐANG VẼ BIỂU ĐỒ HÌNH 5.2 (THAY THẾ HÌNH 7) ---")
        if not os.path.exists('images'): os.makedirs('images')
        
        # Loại bỏ tháng cuối cùng (thường chưa đầy đủ)
        df_filtered = df_monthly_trades.iloc[:-1].copy()
        x_dates = pd.to_datetime(df_filtered['year_month'])
        x_pos = np.arange(len(x_dates))
        width = 0.35
        
        # Chuẩn bị dữ liệu Drawdown theo tháng để khớp trục X
        dd_rl = calculate_drawdown(df_rl['RL_Value'])
        dd_monthly = dd_rl.resample('ME').min() # Resample lấy min drawdown trong tháng
        dd_values = []
        for d in x_dates:
            # Tìm drawdown của tháng tương ứng
            val = dd_monthly[dd_monthly.index.strftime('%Y-%m') == d.strftime('%Y-%m')]
            dd_values.append(val.values[0] if len(val) > 0 else 0)

        # ---------------------------------------------------------------------
        # PHIÊN BẢN 1: CỘT ĐÔI (VÀO/RA) + LINE DRAWDOWN
        # ---------------------------------------------------------------------
        fig_v1, ax1 = plt.subplots(figsize=(16, 7))
        
        # Trục 1: Turnover (Cột)
        ax1.bar(x_pos - width/2, df_filtered['buy_turnover_pct'], width, 
                label='Dòng Tiền VÀO (% NAV)', color='#228B22', alpha=0.9)
        ax1.bar(x_pos + width/2, df_filtered['sell_turnover_pct'], width, 
                label='Dòng Tiền RA (% NAV)', color='#B22222', alpha=0.9)
        
        ax1.set_ylabel("Turnover (% NAV)", fontsize=12)
        ax1.set_ylim(0, 120) # Cố định giới hạn để cột không quá cao
        
        # Trục 2: Drawdown (Line - Tách biệt)
        ax2 = ax1.twinx()
        ax2.plot(x_pos, dd_values, color='black', linewidth=2.5, alpha=0.8, 
                 label='Max Drawdown', marker='o', markersize=4)
        # Tô màu vùng drawdown sâu
        ax2.fill_between(x_pos, 0, dd_values, where=(np.array(dd_values) < -5), color='red', alpha=0.15)
        
        ax2.set_ylabel("Max Drawdown (%)", color='black', fontsize=12)
        ax2.set_ylim(-50, 0) # Drawdown nằm nửa dưới, không che mất cột
        
        # Trang trí
        ax1.set_title("HÌNH 5.2 (V1): DÒNG TIỀN VÀO/RA (% NAV) VÀ DRAWDOWN", fontsize=14, fontweight='bold', pad=15)
        ax1.set_xticks(x_pos)
        ax1.set_xticklabels([d.strftime('%b-%y') for d in x_dates], rotation=45, ha='right')
        
        # Gộp legend
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fancybox=True, framealpha=0.9)
        ax1.grid(True, alpha=0.3)
        
        # Summary text
        summary_txt = (f"Avg Buy: {df_filtered['buy_turnover_pct'].mean():.1f}%\n"
                       f"Avg Sell: {df_filtered['sell_turnover_pct'].mean():.1f}%")
        ax1.text(0.99, 0.98, summary_txt, transform=ax1.transAxes, 
                 bbox=dict(boxstyle="round", fc="white", ec='gray', alpha=0.9), 
                 ha='right', va='top')
        
        plt.tight_layout()
        plt.savefig(f"images/Fig5_2_V1_CashFlow_{METHOD_NAME}.png", dpi=150)
        plt.show()

        # ---------------------------------------------------------------------
        # PHIÊN BẢN 2: NET FLOW (RÒNG) + LINE DRAWDOWN
        # ---------------------------------------------------------------------
        fig_v2, ax1 = plt.subplots(figsize=(16, 7))
        
        # Tính Net Flow
        net_flow = df_filtered['buy_turnover_pct'] - df_filtered['sell_turnover_pct']
        colors = ['#228B22' if x >= 0 else '#B22222' for x in net_flow]
        
        # Trục 1: Net Flow
        ax1.bar(x_pos, net_flow, width=0.5, color=colors, alpha=0.9, label='Net Flow (Mua - Bán)')
        ax1.axhline(y=0, color='gray', linestyle='--', linewidth=1)
        ax1.set_ylabel("Net Flow (% NAV)", fontsize=12)
        ax1.set_ylim(-5, 50)
        
        # Trục 2: Drawdown
        ax2 = ax1.twinx()
        ax2.plot(x_pos, dd_values, color='black', linewidth=2.5, alpha=0.8, 
                 label='Max Drawdown', marker='o', markersize=4)
        ax2.fill_between(x_pos, 0, dd_values, where=(np.array(dd_values) < -5), color='red', alpha=0.15)
        ax2.set_ylabel("Max Drawdown (%)", color='black', fontsize=12)
        ax2.set_ylim(-60, 10)
        
        # Trang trí
        ax1.set_title("HÌNH 5.2 (V2): DÒNG TIỀN RÒNG (NET FLOW) VÀ DRAWDOWN", fontsize=14, fontweight='bold', pad=15)
        ax1.set_xticks(x_pos)
        ax1.set_xticklabels([d.strftime('%b-%y') for d in x_dates], rotation=45, ha='right')
        
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fancybox=True, framealpha=0.9)
        ax1.grid(True, alpha=0.3)
        
        summary_txt2 = (f"Avg Net: {net_flow.mean():.1f}%\n"
                        f"Pos Months: {(net_flow > 0).sum()}\n"
                        f"Neg Months: {(net_flow < 0).sum()}")
        ax1.text(0.99, 0.98, summary_txt2, transform=ax1.transAxes, 
                 bbox=dict(boxstyle="round", fc="white", ec='gray', alpha=0.9), 
                 ha='right', va='top')
        
        plt.tight_layout()
        plt.savefig(f"images/Fig5_2_V2_NetFlow_{METHOD_NAME}.png", dpi=150)
        plt.show()

print("\n--- HOÀN TẤT ---")

--- ĐANG CHUẨN BỊ DỮ LIỆU ĐỂ VẼ HÌNH ---
Đã xuất file metrics.

--- ĐANG VẼ BIỂU ĐỒ ---

--- ĐANG VẼ BIỂU ĐỒ HÌNH 5.2 (THAY THẾ HÌNH 7) ---

--- HOÀN TẤT ---
